# Notebook 05 LightGBM Tuning - Validation Only

Protocol: `docs/NOTEBOOK05_LIGHTGBM_TUNING_PROTOCOL_2026-06-04.md`

Scope: `validation_only`

Research question:

```text
Given Notebook 04D's manual Exit A decision and the fixed official candidate,
does train-inner LightGBM hyperparameter tuning produce a stable validation-only
gain without repeatedly searching the official validation partition?
```

Official candidate:

```text
candidate_id  = stage0_official
label_config  = h03_bps1p5
horizon_k     = 3
threshold_bps = 1.5
feature_set   = price_volume_time
window_size   = 20
```

Notebook 05 parts:

```text
05S = schema smoke, no fitting, no selection
05A = read-only Notebook 04D entry gate and operator Exit A acceptance
05B = train-inner chronological LightGBM HPO
05C = finalist selection from train-inner HPO only
05D = official-validation confirmation of default + train-inner finalists
05E = decision record and allowed wording
```

Hard boundaries:

- no project helper package, prior notebook, or archived helper is imported as
  active logic;
- no holdout/test rows are loaded, transformed, windowed, scored, summarized,
  displayed, or used for wording decisions;
- Notebook 04D is not treated as automatic authorization;
- Notebook 05 requires an explicit operator Exit A acceptance flag before any
  fitting stage;
- HPO uses only chronological train-inner folds inside the official train
  partition;
- official validation confirms the train-inner winner and fixed finalists; it
  must not choose a different official-validation-best winner;
- no confidence threshold or selective/no-trade coverage threshold is selected
  here. That remains reserved for a later separately pre-registered notebook.


In [ ]:
from pathlib import Path
from collections import Counter
import importlib
import json
import math
import random
import shutil
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 160)
warnings.filterwarnings("ignore", message="X does not have valid feature names")

INSTALL_LIGHTGBM_IF_MISSING = False
INSTALL_TORCH_IF_MISSING = False
PYTHON_DEPS_DIR = Path("/content/stage0_python_deps")


def install_package_to_local_target(package_name):
    target_dir = PYTHON_DEPS_DIR
    target_dir.mkdir(parents=True, exist_ok=True)
    base_name = package_name.split("[", 1)[0].replace("-", "_")
    for pattern in (base_name, f"{base_name}-*.dist-info"):
        for path in target_dir.glob(pattern):
            if path.is_dir():
                shutil.rmtree(path)
            elif path.exists():
                path.unlink()
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--upgrade",
        "--target",
        str(target_dir),
        package_name,
    ]
    print("Running dependency install:", " ".join(cmd))
    proc = subprocess.run(cmd, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr, file=sys.stderr)
    if proc.returncode != 0:
        raise RuntimeError(
            f"pip install failed for {package_name} with exit code {proc.returncode}. "
            "Read the pip output above. If Colab reports a package or filesystem "
            "error, restart the runtime and rerun setup cells."
        )
    target_text = str(target_dir)
    if target_text not in sys.path:
        sys.path.insert(0, target_text)
    importlib.invalidate_caches()


def ensure_lightgbm():
    try:
        import lightgbm as lgb
        return lgb
    except (ImportError, OSError) as exc:
        if INSTALL_LIGHTGBM_IF_MISSING:
            print(f"LightGBM import failed before install: {type(exc).__name__}: {exc}")
            install_package_to_local_target("lightgbm")
            sys.modules.pop("lightgbm", None)
            try:
                import lightgbm as lgb
                return lgb
            except (ImportError, OSError) as retry_exc:
                raise RuntimeError(
                    "LightGBM still cannot import after installing into "
                    f"{PYTHON_DEPS_DIR}. Restart the Colab runtime and rerun "
                    "setup cells before Stage 0S."
                ) from retry_exc
        raise RuntimeError(
            "LightGBM import failed. Set INSTALL_LIGHTGBM_IF_MISSING=True and "
            "rerun setup cells. The notebook will install LightGBM into "
            f"{PYTHON_DEPS_DIR} and place that directory first on sys.path."
        ) from exc


def ensure_torch():
    try:
        import torch
        return torch
    except (ImportError, OSError) as exc:
        if INSTALL_TORCH_IF_MISSING:
            print(f"PyTorch import failed before install: {type(exc).__name__}: {exc}")
            install_package_to_local_target("torch")
            sys.modules.pop("torch", None)
            try:
                import torch
                return torch
            except (ImportError, OSError) as retry_exc:
                raise RuntimeError(
                    "PyTorch still cannot import after installing into "
                    f"{PYTHON_DEPS_DIR}. Restart the Colab runtime and rerun "
                    "setup cells before Stage 0B."
                ) from retry_exc
        raise RuntimeError(
            "PyTorch import failed. Set INSTALL_TORCH_IF_MISSING=True and rerun "
            "setup cells before enabling Stage 0B deep models."
        ) from exc


In [ ]:
TICKERS = ("CSCO", "JPM", "KO", "MSFT", "WMT")
RESULT_SCOPE = "validation_only"

INSTALL_LIGHTGBM_IF_MISSING = False
INSTALL_TORCH_IF_MISSING = False

RUN_05A_TO_05E_FULL_PIPELINE = False
RUN_05S_SCHEMA_SMOKE = False
RUN_05A_04D_ENTRY_GATE = False
RUN_05B_TRAIN_INNER_HPO = False
RUN_05C_SELECT_FINALISTS = False
RUN_05D_OFFICIAL_VALIDATION_CONFIRMATION = False
RUN_05E_DECISION_RECORD = False
BACKUP_NOTEBOOK05_TO_GOOGLE_DRIVE = False
NOTEBOOK05_LOCAL_CHECKPOINT_RESUME = True
NOTEBOOK05_DRIVE_CHECKPOINT_BACKUP_EVERY_MINUTES = 30
NOTEBOOK05_DRIVE_CHECKPOINT_BACKUP_EVERY_COMPLETED_UNITS = 25

if RUN_05A_TO_05E_FULL_PIPELINE:
    RUN_05A_04D_ENTRY_GATE = True
    RUN_05B_TRAIN_INNER_HPO = True
    RUN_05C_SELECT_FINALISTS = True
    RUN_05D_OFFICIAL_VALIDATION_CONFIRMATION = True
    RUN_05E_DECISION_RECORD = True

OPERATOR_SELECTED_EXIT = ""
OPERATOR_ACCEPTS_EXIT_A = False
REQUIRED_OPERATOR_EXIT_A = "Exit A - Proceed To 05 LightGBM Tuning"

DRIVE_PROJECT_FOLDER_ID = "15IZ_sOEyyAKmGCUIOv_u17SwQmFX3nG_"
NOTEBOOK04_DRIVE_RESULTS_FOLDER_ID = "1bDhF9glvEwJC_lmp7XRlDGZA4nS-u9Vr"
NOTEBOOK04_DRIVE_RESULTS_FOLDER_NAME = "notebook04_controlled_followup_results"
NOTEBOOK05_DRIVE_BACKUP_FOLDER_NAME = "notebook05_lightgbm_tuning_results"

OFFICIAL_VALIDATION_SEEDS = (606, 707, 808, 909, 1010)

NOTEBOOK05_CANDIDATE = {
    "candidate_id": "stage0_official",
    "label_config": "h03_bps1p5",
    "horizon_k": 3,
    "threshold_bps": 1.5,
    "feature_set": "price_volume_time",
    "window_size": 20,
    "source": "official_stage0_candidate_from_notebook02_and_notebook04d_exit_a",
}

BASELINE_MODELS = ("stratified_dummy", "always_up_dummy")
LIGHTGBM_PROFILES = ("default_lgbm_04", "lightgbm_trial")

ALL_FEATURES = (
    "log_return",
    "close_to_open_return",
    "high_low_range",
    "rolling_volatility_20",
    "normalized_volume_20",
    "rsi_14",
    "bollinger_pctb",
    "normalized_macd_hist",
    "time_of_day_sin",
    "time_of_day_cos",
)

FEATURE_SETS = {
    "price_action_core": (
        "log_return",
        "close_to_open_return",
        "high_low_range",
    ),
    "technical_price": (
        "log_return",
        "high_low_range",
        "rsi_14",
        "bollinger_pctb",
        "normalized_macd_hist",
    ),
    "price_volume_time": ALL_FEATURES,
}

LABEL_CONFIGS = {
    "h03_bps1p5": {"horizon_k": 3, "threshold_bps": 1.5},
    "h09_bps3p0": {"horizon_k": 9, "threshold_bps": 3.0},
    "h24_bps7p5": {"horizon_k": 24, "threshold_bps": 7.5},
}

MAX_TRAIN_ROWS = None
RANDOM_SUBSAMPLE_SEED = 42

LGBM_DEFAULT_PARAMS_04 = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "n_estimators": 200,
    "learning_rate": 0.03,
    "max_depth": 6,
    "num_leaves": 31,
    "subsample": 0.9,
    "subsample_freq": 1,
    "colsample_bytree": 0.9,
    "class_weight": "balanced",
    "verbosity": -1,
}
LGBM_PARAMS = {
    "n_estimators": 200,
    "learning_rate": 0.03,
    "max_depth": 6,
    "num_leaves": 31,
    "subsample": 0.9,
    "subsample_freq": 1,
    "colsample_bytree": 0.9,
}

HPO_METHOD = "random_search"
HPO_BUDGET = 100
HPO_RNG_SEED = 260605
INNER_DUMMY_SEED = 260605
INNER_FOLD_COUNT = 3
MAX_FIT_ROWS_BEFORE_CONFIRMATION = 300
N_FINALISTS = 5
PRIMARY_SELECTION = "train_inner_winner"
EARLY_STOPPING_ROUNDS = 50
MIN_FINALIST_MEDIAN_BEST_ITERATION = 20
PROMOTION_MIN_DELTA_MACRO_F1_VS_DEFAULT = 0.001
PROMOTION_MIN_POSITIVE_TICKER_COUNT = 4
PROMOTION_MAX_TOP_TICKER_GAIN_SHARE = 0.35
PROMOTION_MAX_MACRO_F1_STD = 0.0025

HPO_MAX_ESTIMATORS_CHOICES = (400, 800, 1200, 1600, 2000)
HPO_MAX_DEPTH_CHOICES = (3, 4, 5, 6, 8)
HPO_NUM_LEAVES_CHOICES = (7, 15, 31, 63)
HPO_MIN_CHILD_SAMPLES_CHOICES = (20, 50, 100, 200, 400)

TRAIN_START, TRAIN_END = "1998-01-02", "2013-09-16"
VAL_START, VAL_END = "2013-09-16", "2017-01-25"
CALENDAR_SPLITS = {
    "train": (TRAIN_START, TRAIN_END),
    "validation": (VAL_START, VAL_END),
}

BPS_TO_DECIMAL = 10000.0
BAR_INTERVAL_MINUTES = 5
MARKET_OPEN_MINUTE = 9 * 60 + 30
TRADING_DAY_MINUTES = 390
TIME_OF_DAY_ENCODING_PERIOD_MINUTES = TRADING_DAY_MINUTES + BAR_INTERVAL_MINUTES
MARKET_OPEN = pd.Timestamp("09:30").time()
MARKET_CLOSE = pd.Timestamp("16:00").time()
EXPECTED_COLUMNS = ("timestamp", "open", "high", "low", "close", "volume")
DATA_FILE_SUFFIXES = (".csv", ".txt")
RAW_TXT_COLUMNS = ("Date", "Time", "Open", "High", "Low", "Close", "Volume")

OUTPUT_DIR = Path("/content/notebook05_lightgbm_tuning_results")
PREDICTION_DIR = OUTPUT_DIR / "predictions"
NOTEBOOK04_CONTEXT_DIR = Path("/content/notebook04_controlled_followup_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK04_CONTEXT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILES = {
    "entry": OUTPUT_DIR / "notebook05_entry_decision.json",
    "hpo_search_manifest": OUTPUT_DIR / "notebook05_hpo_search_manifest.csv",
    "inner_fold_manifest": OUTPUT_DIR / "notebook05_inner_fold_manifest.csv",
    "inner_hpo_results": OUTPUT_DIR / "notebook05_inner_hpo_results.csv",
    "inner_hpo_summary": OUTPUT_DIR / "notebook05_inner_hpo_summary.csv",
    "finalists": OUTPUT_DIR / "notebook05_finalists.csv",
    "official_pooled": OUTPUT_DIR / "notebook05_official_validation_pooled.csv",
    "official_per_ticker": OUTPUT_DIR / "notebook05_official_validation_per_ticker.csv",
    "official_summary": OUTPUT_DIR / "notebook05_official_validation_summary.csv",
    "decision": OUTPUT_DIR / "notebook05_decision_record.json",
    "run_manifest": OUTPUT_DIR / "notebook05_run_manifest.json",
    "backup_manifest": OUTPUT_DIR / "notebook05_drive_backup_manifest.json",
}

NOTEBOOK04_FILES = {
    "context": NOTEBOOK04_CONTEXT_DIR / "notebook04_context_checks.json",
    "summary": NOTEBOOK04_CONTEXT_DIR / "notebook04_summary.csv",
    "selective": NOTEBOOK04_CONTEXT_DIR / "notebook04_selective_coverage.csv",
    "decision": NOTEBOOK04_CONTEXT_DIR / "notebook04_decision_matrix.csv",
    "run_manifest": NOTEBOOK04_CONTEXT_DIR / "notebook04_run_manifest.json",
}

RUN_SWITCHES = {
    "RUN_05S_SCHEMA_SMOKE": RUN_05S_SCHEMA_SMOKE,
    "RUN_05A_TO_05E_FULL_PIPELINE": RUN_05A_TO_05E_FULL_PIPELINE,
    "RUN_05A_04D_ENTRY_GATE": RUN_05A_04D_ENTRY_GATE,
    "RUN_05B_TRAIN_INNER_HPO": RUN_05B_TRAIN_INNER_HPO,
    "RUN_05C_SELECT_FINALISTS": RUN_05C_SELECT_FINALISTS,
    "RUN_05D_OFFICIAL_VALIDATION_CONFIRMATION": RUN_05D_OFFICIAL_VALIDATION_CONFIRMATION,
    "RUN_05E_DECISION_RECORD": RUN_05E_DECISION_RECORD,
    "BACKUP_NOTEBOOK05_TO_GOOGLE_DRIVE": BACKUP_NOTEBOOK05_TO_GOOGLE_DRIVE,
    "NOTEBOOK05_LOCAL_CHECKPOINT_RESUME": NOTEBOOK05_LOCAL_CHECKPOINT_RESUME,
    "NOTEBOOK05_DRIVE_CHECKPOINT_BACKUP_EVERY_MINUTES": NOTEBOOK05_DRIVE_CHECKPOINT_BACKUP_EVERY_MINUTES,
    "NOTEBOOK05_DRIVE_CHECKPOINT_BACKUP_EVERY_COMPLETED_UNITS": NOTEBOOK05_DRIVE_CHECKPOINT_BACKUP_EVERY_COMPLETED_UNITS,
}

print("Notebook 05 scope:", RESULT_SCOPE)
print("Notebook 05 candidate:", NOTEBOOK05_CANDIDATE)
print("Run switches:", RUN_SWITCHES)


## Notebook 05 LightGBM Tuning Helpers

This layer adds 04D entry-gate checks, train-inner HPO, finalist selection, official-validation confirmation, and decision records.

In [ ]:
import hashlib


NOTEBOOK05_STATE = {
    "entry_decision": None,
    "backup_folder_id": None,
    "last_drive_checkpoint_utc": None,
}

T_CRITICAL_ONE_SIDED_95_LOCAL = {
    1: 0.0,
    2: 6.314,
    3: 2.920,
    4: 2.353,
    5: 2.132,
    6: 2.015,
    7: 1.943,
    8: 1.895,
    9: 1.860,
    10: 1.833,
}


def stable_hash(values):
    hasher = hashlib.sha256()
    for value in values:
        hasher.update(str(value).encode("utf-8"))
        hasher.update(b"\n")
    return hasher.hexdigest()


def sample_id_hash(values):
    return stable_hash(np.asarray(values).astype(str))


def class_count_fields_05(y_values, prefix):
    y = np.asarray(y_values).astype(int)
    n = int(len(y))
    class0_n = int((y == 0).sum())
    class1_n = int((y == 1).sum())
    positive_rate = float(class1_n / n) if n else np.nan
    return {
        f"{prefix}class0_n": class0_n,
        f"{prefix}class1_n": class1_n,
        f"{prefix}positive_rate": positive_rate,
    }


def series_str_05(values):
    return pd.Series(values).astype(str).reset_index(drop=True)


def series_datetime_str_05(values):
    return pd.Series(pd.to_datetime(values)).astype(str).reset_index(drop=True)


def make_notebook05_sample_ids(owner_values, timestamp_values, y_values, split_name):
    owner = series_str_05(owner_values)
    timestamp = series_datetime_str_05(timestamp_values)
    expected_len = len(y_values)
    if len(owner) != len(timestamp) or len(owner) != expected_len:
        raise ValueError(
            f"Cannot build {split_name}_sample_id; owner/timestamp/y length mismatch: "
            f"owner={len(owner)}, timestamp={len(timestamp)}, y={expected_len}"
        )
    return np.array(
        [
            f"{ticker}|{bar_time}|row{row_index:08d}"
            for row_index, (ticker, bar_time) in enumerate(zip(owner, timestamp))
        ],
        dtype=object,
    )


def t_critical_one_sided_95_local(n):
    n = int(n)
    return T_CRITICAL_ONE_SIDED_95_LOCAL.get(n, 1.645)


def drive_query_literal(value):
    return "'" + str(value).replace("\\", "\\\\").replace("'", "\\'") + "'"


def build_drive_service():
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
    except ImportError as exc:
        raise RuntimeError(
            "Drive API is unavailable. Open this notebook in Google Colab and "
            "authenticate when prompted; do not use Drive mounting for Notebook 05 data."
        ) from exc
    auth.authenticate_user()
    return build("drive", "v3")


def find_latest_drive_file_by_suffix(service, folder_id, filename_suffix):
    escaped_parent = drive_query_literal(folder_id)
    query = f"{escaped_parent} in parents and trashed = false"
    files = []
    page_token = None
    while True:
        response = service.files().list(
            q=query,
            spaces="drive",
            fields="nextPageToken, files(id,name,mimeType,createdTime,modifiedTime)",
            pageSize=100,
            pageToken=page_token,
        ).execute()
        files.extend(
            item
            for item in response.get("files", [])
            if str(item.get("name", "")).endswith(filename_suffix)
        )
        page_token = response.get("nextPageToken")
        if not page_token:
            break
    if not files:
        raise FileNotFoundError(
            f"No Drive file ending with {filename_suffix!r} found in folder "
            f"{NOTEBOOK04_DRIVE_RESULTS_FOLDER_NAME} ({folder_id})."
        )
    return sorted(
        files,
        key=lambda item: (str(item.get("modifiedTime", "")), str(item.get("name", ""))),
        reverse=True,
    )[0]


def download_drive_file(service, file_id, target_path):
    from googleapiclient.http import MediaIoBaseDownload

    target_path = Path(target_path)
    target_path.parent.mkdir(parents=True, exist_ok=True)
    request = service.files().get_media(fileId=file_id)
    with target_path.open("wb") as output:
        downloader = MediaIoBaseDownload(output, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    return target_path


def ensure_latest_notebook04_artifacts_from_drive():
    service = build_drive_service()
    downloaded = {}
    for name, target in NOTEBOOK04_FILES.items():
        found = find_latest_drive_file_by_suffix(service, NOTEBOOK04_DRIVE_RESULTS_FOLDER_ID, target.name)
        download_drive_file(service, found["id"], target)
        downloaded[name] = {
            "drive_id": found["id"],
            "drive_name": found["name"],
            "local_path": str(target),
        }
        print(f"Downloaded Notebook 04 {name}: {found['name']} -> {target}")
    return downloaded


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def required_dataframe(path, name):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Required Notebook 05 input missing: {path}")
    frame = pd.read_csv(path)
    if frame.empty:
        raise ValueError(f"Required Notebook 05 input is empty: {name} at {path}")
    return frame


def artifact_file_hash(path):
    path = Path(path)
    data = path.read_bytes()
    return hashlib.sha256(data).hexdigest()


def validate_fixed_candidate_fields(record, prefix=""):
    expected = NOTEBOOK05_CANDIDATE
    checks = {
        "label_config": expected["label_config"],
        "feature_set": expected["feature_set"],
        "window_size": expected["window_size"],
    }
    for field, expected_value in checks.items():
        value = record.get(field)
        if pd.isna(value):
            raise ValueError(f"{prefix}{field} is missing in Notebook 04 context.")
        if field == "window_size":
            value = int(value)
            expected_value = int(expected_value)
        if value != expected_value:
            raise ValueError(f"{prefix}{field} drifted: {value!r} != {expected_value!r}")
    if "horizon_k" in record and not pd.isna(record["horizon_k"]):
        if int(record["horizon_k"]) != int(expected["horizon_k"]):
            raise ValueError("horizon_k drifted from the official candidate.")
    if "threshold_bps" in record and not pd.isna(record["threshold_bps"]):
        if float(record["threshold_bps"]) != float(expected["threshold_bps"]):
            raise ValueError("threshold_bps drifted from the official candidate.")


def validate_context_official_candidate(context):
    if "official_candidate" not in context:
        raise ValueError("Notebook 04 context is missing official_candidate; Notebook 05 must stop.")
    official_candidate = context["official_candidate"]
    if isinstance(official_candidate, dict):
        validate_fixed_candidate_fields(
            official_candidate,
            prefix="notebook04_context_checks.official_candidate.",
        )
        return
    text = str(official_candidate)
    required_fragments = (
        NOTEBOOK05_CANDIDATE["label_config"],
        NOTEBOOK05_CANDIDATE["feature_set"],
        f"window_size={NOTEBOOK05_CANDIDATE['window_size']}",
    )
    missing = [fragment for fragment in required_fragments if fragment not in text]
    if missing:
        raise ValueError(
            "Notebook 04 context official_candidate does not match Notebook 05 fixed candidate; "
            f"missing fragments: {missing}"
        )


def assert_notebook05_entry_gate(download_if_missing=True):
    if download_if_missing and not all(path.exists() for path in NOTEBOOK04_FILES.values()):
        downloaded = ensure_latest_notebook04_artifacts_from_drive()
    else:
        downloaded = {
            name: {"local_path": str(path), "drive_id": None, "drive_name": path.name}
            for name, path in NOTEBOOK04_FILES.items()
        }

    for name, path in NOTEBOOK04_FILES.items():
        if not path.exists():
            raise FileNotFoundError(f"Required Notebook 04D artifact missing for 05A: {path}")

    context = read_json(NOTEBOOK04_FILES["context"])
    run_manifest = read_json(NOTEBOOK04_FILES["run_manifest"])
    summary = required_dataframe(NOTEBOOK04_FILES["summary"], "notebook04_summary")
    selective = required_dataframe(NOTEBOOK04_FILES["selective"], "notebook04_selective_coverage")
    decision = required_dataframe(NOTEBOOK04_FILES["decision"], "notebook04_decision_matrix")

    if context.get("scope") != RESULT_SCOPE:
        raise ValueError(f"Notebook 04 context scope is not {RESULT_SCOPE}: {context.get('scope')!r}")
    if bool(context.get("holdout_test_authorized")):
        raise ValueError("Notebook 04 context authorizes holdout/test; Notebook 05 must stop.")
    if run_manifest.get("scope") != RESULT_SCOPE:
        raise ValueError(f"Notebook 04 run manifest scope is not {RESULT_SCOPE}: {run_manifest.get('scope')!r}")
    if bool(run_manifest.get("holdout_test_authorized")):
        raise ValueError("Notebook 04 run manifest authorizes holdout/test; Notebook 05 must stop.")
    validate_context_official_candidate(context)

    if OPERATOR_SELECTED_EXIT != REQUIRED_OPERATOR_EXIT_A:
        raise ValueError(
            "Notebook 05 requires OPERATOR_SELECTED_EXIT to be exactly "
            f"{REQUIRED_OPERATOR_EXIT_A!r}."
        )
    if OPERATOR_ACCEPTS_EXIT_A is not True:
        raise ValueError("Notebook 05 requires OPERATOR_ACCEPTS_EXIT_A = True before any fit.")

    lightgbm_rows = summary.loc[summary["model"].astype(str).eq("lightgbm")].copy()
    if len(lightgbm_rows) != 1:
        raise ValueError(f"Expected exactly one Notebook 04 lightgbm summary row, found {len(lightgbm_rows)}.")
    lightgbm_row = lightgbm_rows.iloc[0].to_dict()
    validate_fixed_candidate_fields(lightgbm_row, prefix="notebook04_summary.")
    if not bool(lightgbm_row.get("basic_gate_pass", False)):
        raise ValueError("Notebook 04 LightGBM did not pass the basic gate; Notebook 05 must stop.")
    allowed_stability = {"confirmed_or_improved", "marginal_drop_note_only"}
    if str(lightgbm_row.get("fresh_seed_stability_tag")) not in allowed_stability:
        raise ValueError("Notebook 04 LightGBM fresh-seed stability tag does not authorize Exit A.")

    if "exit" not in decision.columns:
        raise ValueError("Notebook 04 decision matrix is missing the exit column.")
    if not decision["exit"].astype(str).eq(REQUIRED_OPERATOR_EXIT_A).any():
        raise ValueError("Notebook 04 decision matrix does not include Exit A.")
    if "holdout_test_authorized" not in decision.columns:
        raise ValueError("Notebook 04 decision matrix is missing holdout_test_authorized.")
    if decision["holdout_test_authorized"].astype(bool).any():
        raise ValueError("At least one Notebook 04D exit authorizes holdout/test; Notebook 05 must stop.")

    entry = {
        "scope": RESULT_SCOPE,
        "entry_source": "notebook04_04d_decision_gate",
        "created_utc": pd.Timestamp.utcnow().isoformat(),
        "operator_selected_exit": OPERATOR_SELECTED_EXIT,
        "operator_accepts_exit_a": bool(OPERATOR_ACCEPTS_EXIT_A),
        "holdout_test_authorized": False,
        "hpo_authorized": True,
        "authorized_model_family": "lightgbm",
        "authorized_candidate": NOTEBOOK05_CANDIDATE,
        "candidate": NOTEBOOK05_CANDIDATE,
        "notebook04_lightgbm_macro_f1_mean": float(lightgbm_row.get("macro_f1_mean", np.nan)),
        "notebook04_lightgbm_delta_macro_f1_vs_stratified_dummy_mean": float(
            lightgbm_row.get("delta_macro_f1_vs_stratified_dummy_mean", np.nan)
        ),
        "notebook04_selective_rows_read_for_boundary_only": int(len(selective)),
        "notebook04_artifacts": downloaded,
        "notebook04_artifact_hashes": {
            name: artifact_file_hash(path) for name, path in NOTEBOOK04_FILES.items()
        },
    }
    with OUTPUT_FILES["entry"].open("w", encoding="utf-8") as handle:
        json.dump(entry, handle, indent=2)
    NOTEBOOK05_STATE["entry_decision"] = entry
    print("Notebook 05 entry gate passed. HPO is authorized for LightGBM only.")
    return entry


def validate_dataset_candidate(dataset):
    if dataset["label_config"] != NOTEBOOK05_CANDIDATE["label_config"]:
        raise ValueError("Dataset label_config drifted from Notebook 05 candidate.")
    if int(dataset["horizon_k"]) != int(NOTEBOOK05_CANDIDATE["horizon_k"]):
        raise ValueError("Dataset horizon_k drifted from Notebook 05 candidate.")
    if float(dataset["threshold_bps"]) != float(NOTEBOOK05_CANDIDATE["threshold_bps"]):
        raise ValueError("Dataset threshold_bps drifted from Notebook 05 candidate.")
    if dataset["feature_set"] != NOTEBOOK05_CANDIDATE["feature_set"]:
        raise ValueError("Dataset feature_set drifted from Notebook 05 candidate.")
    if int(dataset["window_size"]) != int(NOTEBOOK05_CANDIDATE["window_size"]):
        raise ValueError("Dataset window_size drifted from Notebook 05 candidate.")


def get_notebook05_dataset():
    dataset = get_dataset(
        NOTEBOOK05_CANDIDATE["label_config"],
        NOTEBOOK05_CANDIDATE["feature_set"],
        NOTEBOOK05_CANDIDATE["window_size"],
    )
    validate_dataset_candidate(dataset)
    return dataset


def log_uniform(rng, low, high):
    return float(np.exp(rng.uniform(np.log(low), np.log(high))))


def zero_or_log_uniform(rng, low, high):
    if rng.random() < 0.5:
        return 0.0
    return log_uniform(rng, low, high)


def sample_lgbm_hpo_params(trial_number, rng):
    for _ in range(100):
        max_depth = int(rng.choice(HPO_MAX_DEPTH_CHOICES))
        num_leaves = int(rng.choice(HPO_NUM_LEAVES_CHOICES))
        if max_depth > 0 and num_leaves > 2 ** max_depth:
            continue
        params = {
            "boosting_type": "gbdt",
            "objective": "binary",
            "learning_rate": log_uniform(rng, 0.005, 0.08),
            "max_depth": max_depth,
            "num_leaves": num_leaves,
            "min_child_samples": int(rng.choice(HPO_MIN_CHILD_SAMPLES_CHOICES)),
            "subsample": float(rng.uniform(0.50, 1.00)),
            "subsample_freq": 1,
            "colsample_bytree": float(rng.uniform(0.50, 1.00)),
            "reg_alpha": zero_or_log_uniform(rng, 1e-4, 10.0),
            "reg_lambda": zero_or_log_uniform(rng, 1e-4, 20.0),
            "min_split_gain": float(rng.uniform(0.0, 0.10)),
            "max_estimators": int(rng.choice(HPO_MAX_ESTIMATORS_CHOICES)),
            "early_stopping_rounds": int(EARLY_STOPPING_ROUNDS),
        }
        params["trial_id"] = f"lgbm_hpo_{int(trial_number):03d}"
        return params
    raise RuntimeError(f"Could not sample a valid LightGBM HPO trial for {trial_number}.")


def build_hpo_search_manifest():
    rng = np.random.default_rng(HPO_RNG_SEED)
    rows = []
    for trial_number in range(HPO_BUDGET):
        params = sample_lgbm_hpo_params(trial_number, rng)
        rows.append({
            "trial_id": params["trial_id"],
            "trial_number": int(trial_number),
            "scope": "train_inner_hpo_manifest",
            "hpo_method": HPO_METHOD,
            "hpo_budget": int(HPO_BUDGET),
            "hpo_rng_seed": int(HPO_RNG_SEED),
            "inner_dummy_seed": int(INNER_DUMMY_SEED),
            **{k: v for k, v in params.items() if k != "trial_id"},
        })
    manifest = pd.DataFrame(rows)
    manifest.to_csv(OUTPUT_FILES["hpo_search_manifest"], index=False)
    return manifest


def make_train_inner_folds_05(dataset):
    timestamps = pd.to_datetime(dataset["train_timestamp"])
    dates = pd.Series(timestamps).dt.normalize()
    unique_dates = pd.Series(dates.unique()).sort_values().reset_index(drop=True)
    if len(unique_dates) < INNER_FOLD_COUNT + 1:
        raise ValueError("Not enough train dates for Notebook 05 train-inner folds.")
    chunks = [pd.Series(chunk) for chunk in np.array_split(unique_dates.to_numpy(), INNER_FOLD_COUNT + 1)]
    folds = []
    rows = []
    for fold_index in range(1, INNER_FOLD_COUNT + 1):
        train_dates = pd.concat(chunks[:fold_index], ignore_index=True)
        validation_dates = chunks[fold_index].reset_index(drop=True)
        train_mask = dates.isin(set(train_dates)).to_numpy()
        validation_mask = dates.isin(set(validation_dates)).to_numpy()
        if not train_mask.any() or not validation_mask.any():
            raise ValueError(f"Empty train-inner fold {fold_index}.")
        train_classes = Counter(np.asarray(dataset["y_train"])[train_mask].astype(int))
        validation_classes = Counter(np.asarray(dataset["y_train"])[validation_mask].astype(int))
        if len(validation_classes) < 2:
            raise ValueError(f"Notebook 05 inner fold {fold_index} validation target has one class.")
        if len(train_classes) < 2:
            raise ValueError(f"Notebook 05 inner fold {fold_index} train target has one class.")
        ticker_boundaries = {}
        owners = np.asarray(dataset["train_owner"]).astype(str)
        for ticker in TICKERS:
            ticker_train_dates = dates[train_mask & (owners == ticker)]
            ticker_validation_dates = dates[validation_mask & (owners == ticker)]
            ticker_boundaries[ticker] = {
                "inner_train_start": str(ticker_train_dates.min()) if len(ticker_train_dates) else "",
                "inner_train_end": str(ticker_train_dates.max()) if len(ticker_train_dates) else "",
                "inner_validation_start": str(ticker_validation_dates.min()) if len(ticker_validation_dates) else "",
                "inner_validation_end": str(ticker_validation_dates.max()) if len(ticker_validation_dates) else "",
                "fold_train_n": int((train_mask & (owners == ticker)).sum()),
                "fold_validation_n": int((validation_mask & (owners == ticker)).sum()),
            }
        fold = {
            "inner_fold_id": int(fold_index),
            "train_mask": train_mask,
            "validation_mask": validation_mask,
        }
        folds.append(fold)
        rows.append({
            "inner_fold_id": int(fold_index),
            "scope": "train_inner_validation",
            "inner_train_start": str(train_dates.min()),
            "inner_train_end": str(train_dates.max()),
            "inner_validation_start": str(validation_dates.min()),
            "inner_validation_end": str(validation_dates.max()),
            "fold_train_n": int(train_mask.sum()),
            "fold_validation_n": int(validation_mask.sum()),
            "fold_train_class0_n": int(train_classes.get(0, 0)),
            "fold_train_class1_n": int(train_classes.get(1, 0)),
            "fold_validation_class0_n": int(validation_classes.get(0, 0)),
            "fold_validation_class1_n": int(validation_classes.get(1, 0)),
            "inner_purge_horizon_bars": int(NOTEBOOK05_CANDIDATE["horizon_k"]),
            "ticker_boundaries_json": json.dumps(ticker_boundaries, sort_keys=True),
        })
    fold_manifest = pd.DataFrame(rows)
    fold_manifest.to_csv(OUTPUT_FILES["inner_fold_manifest"], index=False)
    return folds, fold_manifest


def stratified_dummy_predictions_05(y_train, n_rows, seed):
    dummy = DummyClassifier(strategy="stratified", random_state=seed)
    dummy.fit(np.zeros((len(y_train), 1)), y_train)
    return dummy.predict(np.zeros((n_rows, 1))).astype(int)


def always_up_predictions_05(n_rows):
    return np.ones(int(n_rows), dtype=int)


def fit_lightgbm_params_05(x_train, y_train, params, seed, x_eval=None, y_eval=None, use_inner_early_stopping=False):
    lgb = ensure_lightgbm()
    n_estimators = int(params.get("n_estimators", params.get("max_estimators", 200)))
    fit_params = {
        "boosting_type": params.get("boosting_type", "gbdt"),
        "objective": params.get("objective", "binary"),
        "n_estimators": n_estimators,
        "learning_rate": float(params["learning_rate"]),
        "max_depth": int(params["max_depth"]),
        "num_leaves": int(params["num_leaves"]),
        "min_child_samples": int(params.get("min_child_samples", 20)),
        "subsample": float(params.get("subsample", 1.0)),
        "subsample_freq": int(params.get("subsample_freq", 1)),
        "colsample_bytree": float(params.get("colsample_bytree", params.get("feature_fraction", 1.0))),
        "reg_alpha": float(params.get("reg_alpha", 0.0)),
        "reg_lambda": float(params.get("reg_lambda", 0.0)),
        "min_split_gain": float(params.get("min_split_gain", 0.0)),
        "class_weight": params.get("class_weight", "balanced"),
        "random_state": int(seed),
        "verbosity": -1,
    }
    model = lgb.LGBMClassifier(**fit_params)
    callbacks = []
    eval_set = None
    if use_inner_early_stopping:
        if x_eval is None or y_eval is None:
            raise ValueError("Inner early stopping requires an inner validation set.")
        callbacks = [
            lgb.early_stopping(int(params.get("early_stopping_rounds", EARLY_STOPPING_ROUNDS)), verbose=False),
            lgb.log_evaluation(period=0),
        ]
        eval_set = [(x_eval, y_eval)]
    start_fit = time.perf_counter()
    model.fit(
        x_train,
        y_train,
        eval_set=eval_set,
        eval_metric="binary_logloss" if eval_set is not None else None,
        callbacks=callbacks,
    )
    fit_seconds = time.perf_counter() - start_fit
    return model, fit_seconds


def run_one_inner_trial_fold(dataset, trial_params, fold, seed):
    x_train = dataset["x_train"][fold["train_mask"]]
    y_train = dataset["y_train"][fold["train_mask"]]
    x_validation = dataset["x_train"][fold["validation_mask"]]
    y_validation = dataset["y_train"][fold["validation_mask"]]
    dummy_seed = int(INNER_DUMMY_SEED) + int(fold["inner_fold_id"])
    start_predict = None
    try:
        model, fit_seconds = fit_lightgbm_params_05(
            x_train,
            y_train,
            trial_params,
            seed=seed,
            x_eval=x_validation,
            y_eval=y_validation,
            use_inner_early_stopping=True,
        )
        best_iteration = getattr(model, "best_iteration_", None) or int(trial_params["max_estimators"])
        start_predict = time.perf_counter()
        pred = model.predict(x_validation)
        predict_seconds = time.perf_counter() - start_predict
        metrics = evaluate_predictions(y_validation, pred)
        stratified_pred = stratified_dummy_predictions_05(y_train, len(y_validation), dummy_seed)
        always_up_pred = always_up_predictions_05(len(y_validation))
        stratified_metrics = evaluate_predictions(y_validation, stratified_pred)
        always_up_metrics = evaluate_predictions(y_validation, always_up_pred)
        return {
            "run_failed": False,
            "failure_reason": "",
            "fit_seconds": float(fit_seconds),
            "predict_seconds": float(predict_seconds),
            "lightgbm_seed": int(seed),
            "stratified_dummy_seed": int(dummy_seed),
            "best_iteration": int(best_iteration),
            "fold_train_n": int(len(y_train)),
            "fold_validation_n": int(len(y_validation)),
            "fold_train_class0_n": int((y_train == 0).sum()),
            "fold_train_class1_n": int((y_train == 1).sum()),
            "fold_validation_class0_n": int((y_validation == 0).sum()),
            "fold_validation_class1_n": int((y_validation == 1).sum()),
            "stratified_dummy_macro_f1": stratified_metrics["macro_f1"],
            "stratified_dummy_balanced_accuracy": stratified_metrics["balanced_accuracy"],
            "always_up_dummy_macro_f1": always_up_metrics["macro_f1"],
            "always_up_dummy_balanced_accuracy": always_up_metrics["balanced_accuracy"],
            "delta_macro_f1_vs_stratified_dummy": metrics["macro_f1"] - stratified_metrics["macro_f1"],
            "delta_balanced_accuracy_vs_stratified_dummy": metrics["balanced_accuracy"] - stratified_metrics["balanced_accuracy"],
            **metrics,
        }
    except Exception as exc:
        return {
            "run_failed": True,
            "failure_reason": f"{type(exc).__name__}: {exc}",
            "fit_seconds": np.nan,
            "predict_seconds": np.nan,
            "lightgbm_seed": int(seed),
            "stratified_dummy_seed": int(dummy_seed),
            "best_iteration": np.nan,
            "fold_train_n": int(len(y_train)),
            "fold_validation_n": int(len(y_validation)),
            "fold_train_class0_n": int((y_train == 0).sum()),
            "fold_train_class1_n": int((y_train == 1).sum()),
            "fold_validation_class0_n": int((y_validation == 0).sum()),
            "fold_validation_class1_n": int((y_validation == 1).sum()),
            "stratified_dummy_macro_f1": np.nan,
            "stratified_dummy_balanced_accuracy": np.nan,
            "always_up_dummy_macro_f1": np.nan,
            "always_up_dummy_balanced_accuracy": np.nan,
            "delta_macro_f1_vs_stratified_dummy": np.nan,
            "delta_balanced_accuracy_vs_stratified_dummy": np.nan,
            "macro_f1": np.nan,
            "balanced_accuracy": np.nan,
            "accuracy": np.nan,
        }


def summarize_inner_hpo_results(results):
    if results.empty:
        return pd.DataFrame()
    rows = []
    param_columns = [
        "learning_rate",
        "max_depth",
        "num_leaves",
        "min_child_samples",
        "subsample",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda",
        "min_split_gain",
        "max_estimators",
        "early_stopping_rounds",
    ]
    for trial_id, group in results.groupby("trial_id", sort=False):
        successful = group.loc[~group["run_failed"].astype(bool)].copy()
        record = {"trial_id": trial_id, "scope": "train_inner_hpo_summary"}
        for column in param_columns:
            record[column] = group.iloc[0][column]
        record["inner_successful_fold_count"] = int(len(successful))
        record["inner_failed_fold_count"] = int(group["run_failed"].astype(bool).sum())
        if successful.empty:
            for column in (
                "inner_macro_f1_mean",
                "inner_macro_f1_std",
                "inner_macro_f1_min",
                "inner_macro_f1_max",
                "inner_balanced_accuracy_mean",
                "inner_stratified_dummy_macro_f1_mean",
                "inner_delta_macro_f1_vs_stratified_dummy_mean",
                "inner_delta_macro_f1_vs_stratified_dummy_min",
                "inner_lcb_macro_f1",
                "inner_positive_fold_count",
                "median_best_iteration",
            ):
                record[column] = np.nan
            record["eligible_for_finalist"] = False
        else:
            macro_mean = float(successful["macro_f1"].mean())
            macro_std = sample_std(successful["macro_f1"])
            success_count = int(len(successful))
            delta_mean = float(successful["delta_macro_f1_vs_stratified_dummy"].mean())
            delta_min = float(successful["delta_macro_f1_vs_stratified_dummy"].min())
            record.update({
                "inner_macro_f1_mean": macro_mean,
                "inner_macro_f1_std": macro_std,
                "inner_macro_f1_min": float(successful["macro_f1"].min()),
                "inner_macro_f1_max": float(successful["macro_f1"].max()),
                "inner_balanced_accuracy_mean": float(successful["balanced_accuracy"].mean()),
                "inner_stratified_dummy_macro_f1_mean": float(successful["stratified_dummy_macro_f1"].mean()),
                "inner_delta_macro_f1_vs_stratified_dummy_mean": delta_mean,
                "inner_delta_macro_f1_vs_stratified_dummy_min": delta_min,
                "inner_lcb_macro_f1": float(
                    macro_mean
                    - t_critical_one_sided_95_local(success_count) * macro_std / math.sqrt(max(success_count, 1))
                ),
                "inner_positive_fold_count": int((successful["delta_macro_f1_vs_stratified_dummy"] > 0).sum()),
                "median_best_iteration": int(
                    np.clip(
                        round(float(successful["best_iteration"].median())),
                        1,
                        int(group.iloc[0]["max_estimators"]),
                    )
                ),
            })
            record["eligible_for_finalist"] = bool(
                record["inner_successful_fold_count"] == INNER_FOLD_COUNT
                and record["inner_delta_macro_f1_vs_stratified_dummy_mean"] > 0
                and record["inner_delta_macro_f1_vs_stratified_dummy_min"] >= -0.002
                and record["inner_positive_fold_count"] >= 2
                and record["median_best_iteration"] >= MIN_FINALIST_MEDIAN_BEST_ITERATION
            )
        rows.append(record)
    summary = pd.DataFrame(rows)
    summary.to_csv(OUTPUT_FILES["inner_hpo_summary"], index=False)
    return summary


def run_train_inner_hpo_05b():
    assert_notebook05_entry_gate(download_if_missing=True)
    dataset = get_notebook05_dataset()
    search_manifest = build_hpo_search_manifest()
    folds, fold_manifest = make_train_inner_folds_05(dataset)
    if NOTEBOOK05_LOCAL_CHECKPOINT_RESUME and OUTPUT_FILES["inner_hpo_results"].exists():
        existing_results = pd.read_csv(OUTPUT_FILES["inner_hpo_results"])
        missing = {"trial_id", "inner_fold_id"} - set(existing_results.columns)
        if missing:
            raise ValueError(f"Existing 05B checkpoint is missing columns: {sorted(missing)}")
        rows = existing_results.to_dict("records")
    else:
        rows = []
    completed = {
        (str(row["trial_id"]), int(row["inner_fold_id"]))
        for row in rows
    }
    for _, trial in search_manifest.iterrows():
        trial_params = trial.to_dict()
        seed = int(HPO_RNG_SEED) + int(trial["trial_number"])
        for fold in folds:
            key = (str(trial["trial_id"]), int(fold["inner_fold_id"]))
            if key in completed:
                print("05B checkpoint skip", trial["trial_id"], "fold", fold["inner_fold_id"])
                continue
            print("05B", trial["trial_id"], "fold", fold["inner_fold_id"])
            result = run_one_inner_trial_fold(dataset, trial_params, fold, seed=seed)
            rows.append({
                "trial_id": trial["trial_id"],
                "trial_number": int(trial["trial_number"]),
                "inner_fold_id": int(fold["inner_fold_id"]),
                "scope": "train_inner_validation",
                **{column: trial[column] for column in search_manifest.columns if column not in {"scope"}},
                **result,
            })
            completed.add(key)
            pd.DataFrame(rows).to_csv(OUTPUT_FILES["inner_hpo_results"], index=False)
            maybe_backup_notebook05_checkpoint(
                "checkpoint_05B_train_inner_hpo",
                completed_units=len(completed),
                include_predictions=False,
            )
    results = pd.DataFrame(rows)
    results.to_csv(OUTPUT_FILES["inner_hpo_results"], index=False)
    summary = summarize_inner_hpo_results(results)
    write_run_manifest()
    return search_manifest, fold_manifest, results, summary


def select_finalists_05c():
    if not OUTPUT_FILES["inner_hpo_summary"].exists():
        raise FileNotFoundError("Notebook 05 inner HPO summary is missing. Run 05B first.")
    summary = pd.read_csv(OUTPUT_FILES["inner_hpo_summary"])
    if summary.empty:
        finalists = pd.DataFrame()
        finalists.to_csv(OUTPUT_FILES["finalists"], index=False)
        return finalists
    eligible = summary.loc[summary["eligible_for_finalist"].astype(bool)].copy()
    if eligible.empty:
        finalists = pd.DataFrame()
        finalists.to_csv(OUTPUT_FILES["finalists"], index=False)
        return finalists
    eligible["max_depth_positive_sort"] = eligible["max_depth"].apply(lambda value: int(value) if int(value) > 0 else 999)
    eligible = eligible.sort_values(
        [
            "inner_lcb_macro_f1",
            "inner_macro_f1_std",
            "num_leaves",
            "max_depth_positive_sort",
            "median_best_iteration",
            "trial_id",
        ],
        ascending=[False, True, True, True, True, True],
    ).head(N_FINALISTS).copy()
    eligible.insert(0, "finalist_rank", np.arange(1, len(eligible) + 1))
    eligible["profile_id"] = eligible["trial_id"].map(lambda value: str(value).replace("lgbm_hpo_", "lightgbm_trial_"))
    eligible["profile_role"] = eligible["finalist_rank"].map(lambda rank: "train_inner_winner" if rank == 1 else "train_inner_finalist")
    eligible["selected_profile_source"] = PRIMARY_SELECTION
    eligible["holdout_test_authorized"] = False
    eligible.to_csv(OUTPUT_FILES["finalists"], index=False)
    return eligible


def lgbm_params_from_profile(profile):
    if profile["profile_id"] == "default_lgbm_04":
        params = dict(LGBM_DEFAULT_PARAMS_04)
        params["n_estimators"] = int(params["n_estimators"])
        return params
    params = {
        "boosting_type": "gbdt",
        "objective": "binary",
        "learning_rate": float(profile["learning_rate"]),
        "max_depth": int(profile["max_depth"]),
        "num_leaves": int(profile["num_leaves"]),
        "min_child_samples": int(profile["min_child_samples"]),
        "subsample": float(profile["subsample"]),
        "subsample_freq": 1,
        "colsample_bytree": float(profile["colsample_bytree"]),
        "reg_alpha": float(profile["reg_alpha"]),
        "reg_lambda": float(profile["reg_lambda"]),
        "min_split_gain": float(profile["min_split_gain"]),
        "class_weight": "balanced",
        "verbosity": -1,
        "n_estimators": int(profile["median_best_iteration"]),
    }
    return params


def profile_rows_for_confirmation():
    if not OUTPUT_FILES["finalists"].exists():
        raise FileNotFoundError("Notebook 05 finalists artifact is missing. Run 05C before 05D.")
    rows = [{
        "profile_id": "default_lgbm_04",
        "profile_role": "default_lgbm_04",
        "selected_by_train_inner": False,
        "trial_id": "",
        "median_best_iteration": int(LGBM_DEFAULT_PARAMS_04["n_estimators"]),
        **LGBM_DEFAULT_PARAMS_04,
    }]
    finalists = pd.read_csv(OUTPUT_FILES["finalists"])
    if not finalists.empty:
        for _, row in finalists.iterrows():
            record = row.to_dict()
            record["selected_by_train_inner"] = bool(record.get("profile_role") == "train_inner_winner")
            rows.append(record)
    return pd.DataFrame(rows)


def save_probability_artifact_05(dataset, profile_id, seed, y_pred, prob_up):
    prob_up = np.asarray(prob_up, dtype=float)
    confidence = np.maximum(prob_up, 1.0 - prob_up)
    payload = {
        "validation_sample_id": np.asarray(dataset["validation_sample_id"]).astype(str),
        "ticker": np.asarray(dataset["validation_owner"]).astype(str),
        "timestamp": dataset["validation_timestamp"].astype("datetime64[ns]").astype(str),
        "y_true": dataset["y_validation"].astype(int),
        "y_pred": np.asarray(y_pred, dtype=int),
        "prob_up": prob_up,
        "confidence": confidence,
    }
    artifact_path = PREDICTION_DIR / f"{profile_id}__seed{int(seed)}.npz"
    np.savez_compressed(artifact_path, **payload)
    return artifact_path


def metric_row_05(y_true, y_pred, stratified_pred, always_up_pred):
    metrics = evaluate_predictions(y_true, y_pred)
    stratified_metrics = evaluate_predictions(y_true, stratified_pred)
    always_up_metrics = evaluate_predictions(y_true, always_up_pred)
    metrics.update({
        "stratified_dummy_macro_f1": stratified_metrics["macro_f1"],
        "stratified_dummy_balanced_accuracy": stratified_metrics["balanced_accuracy"],
        "delta_macro_f1_vs_stratified_dummy": metrics["macro_f1"] - stratified_metrics["macro_f1"],
        "delta_balanced_accuracy_vs_stratified_dummy": metrics["balanced_accuracy"] - stratified_metrics["balanced_accuracy"],
        "delta_macro_f1_vs_stratified_dummy_same_rows": metrics["macro_f1"] - stratified_metrics["macro_f1"],
        "delta_balanced_accuracy_vs_stratified_dummy_same_rows": metrics["balanced_accuracy"] - stratified_metrics["balanced_accuracy"],
        "always_up_dummy_macro_f1": always_up_metrics["macro_f1"],
        "always_up_dummy_balanced_accuracy": always_up_metrics["balanced_accuracy"],
        "delta_macro_f1_vs_always_up_dummy": metrics["macro_f1"] - always_up_metrics["macro_f1"],
        "delta_balanced_accuracy_vs_always_up_dummy": metrics["balanced_accuracy"] - always_up_metrics["balanced_accuracy"],
        "delta_macro_f1_vs_always_up_dummy_same_rows": metrics["macro_f1"] - always_up_metrics["macro_f1"],
        "delta_balanced_accuracy_vs_always_up_dummy_same_rows": metrics["balanced_accuracy"] - always_up_metrics["balanced_accuracy"],
    })
    return metrics


def official_profile_fit_rows(dataset, profile, seed):
    profile = dict(profile)
    profile_id = str(profile["profile_id"])
    profile_role = str(profile["profile_role"])
    params = lgbm_params_from_profile(profile)
    x_train = dataset["x_train"]
    y_train = dataset["y_train"]
    x_validation = dataset["x_validation"]
    y_validation = dataset["y_validation"]
    train_owner = np.asarray(dataset["train_owner"]).astype(str)
    pooled_train_class_counts = class_count_fields_05(y_train, "train_")
    model, fit_seconds = fit_lightgbm_params_05(
        x_train,
        y_train,
        params,
        seed=seed,
        use_inner_early_stopping=False,
    )
    start_predict = time.perf_counter()
    y_pred = model.predict(x_validation).astype(int)
    prob_up = model.predict_proba(x_validation)[:, 1].astype(float)
    predict_seconds = time.perf_counter() - start_predict
    artifact_path = save_probability_artifact_05(dataset, profile_id, seed, y_pred, prob_up)
    stratified_pred = stratified_dummy_predictions_05(y_train, len(y_validation), seed)
    always_up_pred = always_up_predictions_05(len(y_validation))
    pooled_metrics = metric_row_05(y_validation, y_pred, stratified_pred, always_up_pred)
    sample_hash = sample_id_hash(dataset["validation_sample_id"])
    per_ticker_rows = []
    for ticker in TICKERS:
        mask = dataset["validation_owner"] == ticker
        if not mask.any():
            continue
        train_mask = train_owner == ticker
        ticker_train_class_counts = class_count_fields_05(y_train[train_mask], "train_")
        per_ticker_metrics = metric_row_05(
            y_validation[mask],
            y_pred[mask],
            stratified_pred[mask],
            always_up_pred[mask],
        )
        per_ticker_rows.append({
            "stage": "05D_official_validation_confirmation",
            "candidate_id": NOTEBOOK05_CANDIDATE["candidate_id"],
            "profile_id": profile_id,
            "profile_role": profile_role,
            "seed": int(seed),
            "label_config": dataset["label_config"],
            "horizon_k": int(dataset["horizon_k"]),
            "threshold_bps": float(dataset["threshold_bps"]),
            "feature_set": dataset["feature_set"],
            "window_size": int(dataset["window_size"]),
            "ticker_or_pooled": ticker,
            "scope": RESULT_SCOPE,
            "n": int(mask.sum()),
            "train_n": int((dataset["train_owner"] == ticker).sum()),
            "validation_n": int(mask.sum()),
            **ticker_train_class_counts,
            "validation_sample_id_hash": sample_hash,
            "sample_id_mismatch_count": 0,
            "selected_by_train_inner": bool(profile.get("selected_by_train_inner", False)),
            "official_validation_used_for_selection": False,
            "selected_profile_source": PRIMARY_SELECTION,
            "fit_seconds": float(fit_seconds),
            "predict_seconds": float(predict_seconds),
            "prediction_artifact": str(artifact_path),
            **per_ticker_metrics,
        })
    positive = [row["delta_macro_f1_vs_stratified_dummy"] for row in per_ticker_rows if row["delta_macro_f1_vs_stratified_dummy"] > 0]
    positive_ticker_count = int(len(positive))
    top_ticker_gain_share = float(max(positive) / sum(positive)) if positive else 0.0
    for row in per_ticker_rows:
        row["positive_ticker_count"] = positive_ticker_count
        row["top_ticker_gain_share"] = top_ticker_gain_share
    pooled_row = {
        "stage": "05D_official_validation_confirmation",
        "candidate_id": NOTEBOOK05_CANDIDATE["candidate_id"],
        "profile_id": profile_id,
        "profile_role": profile_role,
        "seed": int(seed),
        "label_config": dataset["label_config"],
        "horizon_k": int(dataset["horizon_k"]),
        "threshold_bps": float(dataset["threshold_bps"]),
        "feature_set": dataset["feature_set"],
        "window_size": int(dataset["window_size"]),
        "ticker_or_pooled": "pooled",
        "scope": RESULT_SCOPE,
        "n": int(len(y_validation)),
        "train_n": int(len(y_train)),
        "validation_n": int(len(y_validation)),
        **pooled_train_class_counts,
        "validation_sample_id_hash": sample_hash,
        "sample_id_mismatch_count": 0,
        "selected_by_train_inner": bool(profile.get("selected_by_train_inner", False)),
        "official_validation_used_for_selection": False,
        "selected_profile_source": PRIMARY_SELECTION,
        "fit_seconds": float(fit_seconds),
        "predict_seconds": float(predict_seconds),
        "prediction_artifact": str(artifact_path),
        "positive_ticker_count": positive_ticker_count,
        "top_ticker_gain_share": top_ticker_gain_share,
        **pooled_metrics,
    }
    return pooled_row, per_ticker_rows


def dummy_official_rows_05(dataset, model_name, seed):
    y_train = dataset["y_train"]
    y_validation = dataset["y_validation"]
    train_owner = np.asarray(dataset["train_owner"]).astype(str)
    pooled_train_class_counts = class_count_fields_05(y_train, "train_")
    if model_name == "stratified_dummy":
        y_pred = stratified_dummy_predictions_05(y_train, len(y_validation), seed)
    elif model_name == "always_up_dummy":
        y_pred = always_up_predictions_05(len(y_validation))
    else:
        raise ValueError(f"Unsupported dummy model: {model_name}")
    stratified_pred = stratified_dummy_predictions_05(y_train, len(y_validation), seed)
    always_up_pred = always_up_predictions_05(len(y_validation))
    metrics = metric_row_05(y_validation, y_pred, stratified_pred, always_up_pred)
    sample_hash = sample_id_hash(dataset["validation_sample_id"])
    pooled_row = {
        "stage": "05D_official_validation_confirmation",
        "candidate_id": NOTEBOOK05_CANDIDATE["candidate_id"],
        "profile_id": model_name,
        "profile_role": model_name,
        "seed": int(seed),
        "label_config": dataset["label_config"],
        "horizon_k": int(dataset["horizon_k"]),
        "threshold_bps": float(dataset["threshold_bps"]),
        "feature_set": dataset["feature_set"],
        "window_size": int(dataset["window_size"]),
        "ticker_or_pooled": "pooled",
        "scope": RESULT_SCOPE,
        "n": int(len(y_validation)),
        "train_n": int(len(y_train)),
        "validation_n": int(len(y_validation)),
        **pooled_train_class_counts,
        "validation_sample_id_hash": sample_hash,
        "sample_id_mismatch_count": 0,
        "selected_by_train_inner": False,
        "official_validation_used_for_selection": False,
        "selected_profile_source": "dummy_baseline",
        "fit_seconds": 0.0,
        "predict_seconds": 0.0,
        "prediction_artifact": "",
        "positive_ticker_count": np.nan,
        "top_ticker_gain_share": np.nan,
        **metrics,
    }
    per_ticker_rows = []
    for ticker in TICKERS:
        mask = dataset["validation_owner"] == ticker
        if not mask.any():
            continue
        train_mask = train_owner == ticker
        ticker_train_class_counts = class_count_fields_05(y_train[train_mask], "train_")
        per_ticker_metrics = metric_row_05(
            y_validation[mask],
            y_pred[mask],
            stratified_pred[mask],
            always_up_pred[mask],
        )
        per_ticker_rows.append({
            **pooled_row,
            "ticker_or_pooled": ticker,
            "n": int(mask.sum()),
            "train_n": int((dataset["train_owner"] == ticker).sum()),
            "validation_n": int(mask.sum()),
            **ticker_train_class_counts,
            **per_ticker_metrics,
        })
    return pooled_row, per_ticker_rows


def summarize_official_validation_05d(pooled):
    if pooled.empty:
        return pd.DataFrame()
    rows = []
    for profile_id, group in pooled.groupby("profile_id", sort=False):
        successful = group.copy()
        record = {
            "profile_id": profile_id,
            "profile_role": str(group.iloc[0]["profile_role"]),
            "scope": RESULT_SCOPE,
            "seed_count": int(successful["seed"].nunique()),
            "selected_by_train_inner": bool(successful["selected_by_train_inner"].astype(bool).any()),
            "selected_profile_source": str(group.iloc[0]["selected_profile_source"]),
            "validation_sample_id_hash": str(group.iloc[0]["validation_sample_id_hash"]),
            "sample_id_mismatch_count": int(successful["sample_id_mismatch_count"].sum()),
            "macro_f1_mean": float(successful["macro_f1"].mean()),
            "macro_f1_std": sample_std(successful["macro_f1"]),
            "balanced_accuracy_mean": float(successful["balanced_accuracy"].mean()),
            "stratified_dummy_macro_f1_mean": float(successful["stratified_dummy_macro_f1"].mean()),
            "delta_macro_f1_vs_stratified_dummy_mean": float(successful["delta_macro_f1_vs_stratified_dummy"].mean()),
            "always_up_dummy_macro_f1_mean": float(successful["always_up_dummy_macro_f1"].mean()),
            "delta_macro_f1_vs_always_up_dummy_mean": float(successful["delta_macro_f1_vs_always_up_dummy"].mean()),
            "positive_ticker_count": np.nan,
            "top_ticker_gain_share": np.nan,
        }
        record["macro_f1_lcb_95"] = float(
            record["macro_f1_mean"]
            - t_critical_one_sided_95_local(record["seed_count"])
            * record["macro_f1_std"]
            / math.sqrt(max(record["seed_count"], 1))
        )
        concentration = successful["positive_ticker_count"].dropna()
        if not concentration.empty:
            record["positive_ticker_count"] = int(round(float(concentration.mean())))
        gain_share = successful["top_ticker_gain_share"].dropna()
        if not gain_share.empty:
            record["top_ticker_gain_share"] = float(gain_share.mean())
        rows.append(record)
    summary = pd.DataFrame(rows)
    default_rows = summary.loc[summary["profile_id"].eq("default_lgbm_04")]
    if not default_rows.empty:
        default_macro = float(default_rows.iloc[0]["macro_f1_mean"])
        summary["delta_macro_f1_vs_default_lgbm_04"] = summary["macro_f1_mean"] - default_macro
    else:
        summary["delta_macro_f1_vs_default_lgbm_04"] = np.nan
    summary = summary.sort_values("macro_f1_mean", ascending=False).reset_index(drop=True)
    summary["official_validation_rank_by_macro_f1"] = np.arange(1, len(summary) + 1)
    summary["official_validation_diagnostic_rank_by_macro_f1"] = np.arange(1, len(summary) + 1)
    official_best = summary.iloc[0]["profile_id"] if not summary.empty else ""
    train_inner_rows = summary.loc[summary["selected_by_train_inner"].astype(bool)]
    train_inner_winner = train_inner_rows.iloc[0]["profile_id"] if not train_inner_rows.empty else ""
    summary["selected_by_official_validation"] = summary["profile_id"].eq(official_best)
    summary["official_validation_diagnostic_best"] = summary["profile_id"].eq(official_best)
    summary["official_validation_ranking_disagrees_with_train_inner"] = bool(
        train_inner_winner and official_best != train_inner_winner
    )
    summary.to_csv(OUTPUT_FILES["official_summary"], index=False)
    return summary


def run_official_validation_confirmation_05d():
    assert_notebook05_entry_gate(download_if_missing=True)
    dataset = get_notebook05_dataset()
    profiles = profile_rows_for_confirmation()
    if NOTEBOOK05_LOCAL_CHECKPOINT_RESUME and OUTPUT_FILES["official_pooled"].exists():
        existing_pooled = pd.read_csv(OUTPUT_FILES["official_pooled"])
        missing = {"profile_id", "seed"} - set(existing_pooled.columns)
        if missing:
            raise ValueError(f"Existing 05D pooled checkpoint is missing columns: {sorted(missing)}")
        pooled_rows = existing_pooled.to_dict("records")
    else:
        pooled_rows = []
    if NOTEBOOK05_LOCAL_CHECKPOINT_RESUME and OUTPUT_FILES["official_per_ticker"].exists():
        existing_per_ticker = pd.read_csv(OUTPUT_FILES["official_per_ticker"])
        missing = {"profile_id", "seed", "ticker_or_pooled"} - set(existing_per_ticker.columns)
        if missing:
            raise ValueError(f"Existing 05D per-ticker checkpoint is missing columns: {sorted(missing)}")
        per_ticker_rows = existing_per_ticker.to_dict("records")
    else:
        per_ticker_rows = []
    completed = {
        (str(row["profile_id"]), int(row["seed"]))
        for row in pooled_rows
    }

    def write_05d_checkpoint():
        pooled_checkpoint = pd.DataFrame(pooled_rows)
        per_ticker_checkpoint = pd.DataFrame(per_ticker_rows)
        pooled_checkpoint.to_csv(OUTPUT_FILES["official_pooled"], index=False)
        per_ticker_checkpoint.to_csv(OUTPUT_FILES["official_per_ticker"], index=False)
        summarize_official_validation_05d(pooled_checkpoint)
        write_run_manifest()

    for seed in OFFICIAL_VALIDATION_SEEDS:
        for model_name in BASELINE_MODELS:
            key = (str(model_name), int(seed))
            if key in completed:
                print("05D checkpoint skip", model_name, "seed", seed)
                continue
            print("05D", model_name, "seed", seed)
            pooled_row, ticker_rows = dummy_official_rows_05(dataset, model_name, seed)
            pooled_rows.append(pooled_row)
            per_ticker_rows.extend(ticker_rows)
            completed.add(key)
            write_05d_checkpoint()
            maybe_backup_notebook05_checkpoint(
                "checkpoint_05D_official_validation_confirmation",
                completed_units=len(completed),
                include_predictions=True,
            )
        for _, profile in profiles.iterrows():
            key = (str(profile["profile_id"]), int(seed))
            if key in completed:
                print("05D checkpoint skip", profile["profile_id"], "seed", seed)
                continue
            print("05D", profile["profile_id"], "seed", seed)
            pooled_row, ticker_rows = official_profile_fit_rows(dataset, profile.to_dict(), seed)
            pooled_rows.append(pooled_row)
            per_ticker_rows.extend(ticker_rows)
            completed.add(key)
            write_05d_checkpoint()
            maybe_backup_notebook05_checkpoint(
                "checkpoint_05D_official_validation_confirmation",
                completed_units=len(completed),
                include_predictions=True,
            )
    pooled = pd.DataFrame(pooled_rows)
    per_ticker = pd.DataFrame(per_ticker_rows)
    summary = summarize_official_validation_05d(pooled)
    pooled.to_csv(OUTPUT_FILES["official_pooled"], index=False)
    per_ticker.to_csv(OUTPUT_FILES["official_per_ticker"], index=False)
    write_run_manifest()
    return pooled, per_ticker, summary


def official_validation_status_05e(official_summary, selected_profile_id):
    if official_summary.empty:
        raise ValueError("Notebook 05 official validation summary is empty. Run 05D before 05E.")
    selected_rows = official_summary.loc[official_summary["profile_id"].astype(str).eq(str(selected_profile_id))]
    if selected_rows.empty:
        raise ValueError(f"Selected train-inner profile is missing from official validation summary: {selected_profile_id}")
    default_rows = official_summary.loc[official_summary["profile_id"].astype(str).eq("default_lgbm_04")]
    if default_rows.empty:
        raise ValueError("default_lgbm_04 is missing from official validation summary.")
    selected = selected_rows.iloc[0]
    default = default_rows.iloc[0]
    selected_delta_vs_default = float(selected["delta_macro_f1_vs_default_lgbm_04"])
    selected_macro_std = float(selected["macro_f1_std"])
    default_macro_std = float(default["macro_f1_std"])
    status_checks = {
        "all_official_validation_seeds_present": int(selected["seed_count"]) == len(OFFICIAL_VALIDATION_SEEDS),
        "beats_stratified_dummy_mean": float(selected["delta_macro_f1_vs_stratified_dummy_mean"]) > 0,
        "lcb_beats_stratified_dummy_mean": float(selected["macro_f1_lcb_95"]) > float(selected["stratified_dummy_macro_f1_mean"]),
        "beats_default_by_practical_margin": selected_delta_vs_default >= PROMOTION_MIN_DELTA_MACRO_F1_VS_DEFAULT,
        "enough_positive_tickers": int(selected["positive_ticker_count"]) >= PROMOTION_MIN_POSITIVE_TICKER_COUNT,
        "not_ticker_concentrated": float(selected["top_ticker_gain_share"]) <= PROMOTION_MAX_TOP_TICKER_GAIN_SHARE,
        "seed_std_within_limit": selected_macro_std <= max(PROMOTION_MAX_MACRO_F1_STD, 3.0 * default_macro_std),
    }
    if all(status_checks.values()):
        return "promote_train_inner_winner", status_checks
    if selected_delta_vs_default < PROMOTION_MIN_DELTA_MACRO_F1_VS_DEFAULT:
        return "no_practical_tuning_gain", status_checks
    official_best_rows = official_summary.loc[official_summary["selected_by_official_validation"].astype(bool)]
    official_best = str(official_best_rows.iloc[0]["profile_id"]) if not official_best_rows.empty else ""
    if official_best == "default_lgbm_04":
        return "retain_default_lgbm_04", status_checks
    return "validation_rejects_train_inner_winner", status_checks


def write_decision_record_05e():
    entry = NOTEBOOK05_STATE.get("entry_decision")
    if entry is None and OUTPUT_FILES["entry"].exists():
        entry = read_json(OUTPUT_FILES["entry"])
    finalists = pd.read_csv(OUTPUT_FILES["finalists"]) if OUTPUT_FILES["finalists"].exists() else pd.DataFrame()
    if not OUTPUT_FILES["official_summary"].exists():
        raise FileNotFoundError("Notebook 05 official validation summary is missing. Run 05D before 05E.")
    official_summary = pd.read_csv(OUTPUT_FILES["official_summary"])
    n_finalists_found = int(len(finalists)) if not finalists.empty else 0
    finalist_count_below_target = bool(n_finalists_found < int(N_FINALISTS))
    promotion_checks = {}
    if finalists.empty:
        decision = "no_train_inner_hpo_candidate"
        selected_profile_id = "default_lgbm_04"
        selected_profile_source = "default_context_only"
        official_validation_status = "no_train_inner_hpo_candidate"
    else:
        winner = finalists.sort_values("finalist_rank").iloc[0]
        selected_profile_id = str(winner["profile_id"])
        selected_profile_source = PRIMARY_SELECTION
        official_validation_status, promotion_checks = official_validation_status_05e(
            official_summary,
            selected_profile_id,
        )
        decision = official_validation_status
    official_best_profile_id = ""
    official_disagrees = False
    if not official_summary.empty:
        best = official_summary.sort_values("official_validation_rank_by_macro_f1").iloc[0]
        official_best_profile_id = str(best["profile_id"])
        official_disagrees = bool(official_best_profile_id != selected_profile_id)
    downstream_primary_profile_id = (
        selected_profile_id
        if official_validation_status == "promote_train_inner_winner"
        else "default_lgbm_04"
    )
    record = {
        "scope": RESULT_SCOPE,
        "created_utc": pd.Timestamp.utcnow().isoformat(),
        "decision": decision,
        "selected_profile_id": selected_profile_id,
        "selected_profile_source": selected_profile_source,
        "train_inner_selected_profile_id": selected_profile_id,
        "official_validation_status": official_validation_status,
        "promotion_checks": promotion_checks,
        "downstream_primary_profile_id": downstream_primary_profile_id,
        "retained_default_lgbm_04": bool(downstream_primary_profile_id == "default_lgbm_04"),
        "n_finalists_found": n_finalists_found,
        "n_finalists_target": int(N_FINALISTS),
        "finalist_count_below_target": finalist_count_below_target,
        "official_validation_best_profile_id": official_best_profile_id,
        "official_validation_ranking_disagrees_with_train_inner": official_disagrees,
        "holdout_test_authorized": False,
        "selective_threshold_selected": False,
        "allowed_wording": [
            "Notebook 05 performs validation-only LightGBM hyperparameter tuning under a train-inner chronological HPO design.",
            "The selected tuned configuration is the train-inner HPO winner, not the official-validation-best finalist.",
            "Official validation assigns a pre-registered promotion, retention, or rejection status without selecting a new winner.",
            "Notebook 05 does not authorize holdout/test evaluation.",
        ],
        "forbidden_wording": [
            "The tuned model is final.",
            "The tuned model is holdout-ready.",
            "The tuned model significantly beats LogReg.",
            "The tuned model proves LightGBM is superior to deep learning.",
            "The official-validation-best finalist is selected.",
            "Selective coverage is now the final trading threshold.",
        ],
        "candidate": NOTEBOOK05_CANDIDATE,
        "entry_decision": entry,
        "budget_tracker": budget_tracker_05(),
    }
    with OUTPUT_FILES["decision"].open("w", encoding="utf-8") as handle:
        json.dump(record, handle, indent=2)
    write_run_manifest()
    return record


def budget_tracker_05():
    inner_results = pd.read_csv(OUTPUT_FILES["inner_hpo_results"]) if OUTPUT_FILES["inner_hpo_results"].exists() else pd.DataFrame()
    finalists = pd.read_csv(OUTPUT_FILES["finalists"]) if OUTPUT_FILES["finalists"].exists() else pd.DataFrame()
    official_pooled = pd.read_csv(OUTPUT_FILES["official_pooled"]) if OUTPUT_FILES["official_pooled"].exists() else pd.DataFrame()
    return {
        "hpo_method": HPO_METHOD,
        "hpo_budget": int(HPO_BUDGET),
        "inner_fold_count": int(INNER_FOLD_COUNT),
        "train_inner_lightgbm_fits_planned": int(HPO_BUDGET * INNER_FOLD_COUNT),
        "train_inner_lightgbm_fits_completed": int(len(inner_results)) if not inner_results.empty else 0,
        "train_inner_hpo_complete": bool(len(inner_results) == HPO_BUDGET * INNER_FOLD_COUNT),
        "n_finalists_available": int(len(finalists)) if not finalists.empty else 0,
        "official_validation_lightgbm_rows_planned": int((1 + N_FINALISTS) * len(OFFICIAL_VALIDATION_SEEDS)),
        "official_validation_lightgbm_rows_completed": int(
            len(official_pooled.loc[~official_pooled["profile_id"].isin(BASELINE_MODELS)])
        ) if not official_pooled.empty else 0,
        "official_validation_dummy_rows_completed": int(
            len(official_pooled.loc[official_pooled["profile_id"].isin(BASELINE_MODELS)])
        ) if not official_pooled.empty else 0,
        "holdout_test_authorized": False,
    }


def write_run_manifest():
    manifest = {
        "scope": RESULT_SCOPE,
        "created_utc": pd.Timestamp.utcnow().isoformat(),
        "candidate": NOTEBOOK05_CANDIDATE,
        "run_switches": RUN_SWITCHES,
        "operator_selected_exit": OPERATOR_SELECTED_EXIT,
        "operator_accepts_exit_a": bool(OPERATOR_ACCEPTS_EXIT_A),
        "primary_selection": PRIMARY_SELECTION,
        "holdout_test_authorized": False,
        "selective_threshold_selected": False,
        "budget_tracker": budget_tracker_05(),
        "output_files": {name: str(path) for name, path in OUTPUT_FILES.items()},
    }
    with OUTPUT_FILES["run_manifest"].open("w", encoding="utf-8") as handle:
        json.dump(manifest, handle, indent=2)
    return manifest


def find_or_create_drive_folder(service, folder_name, parent_id):
    escaped_parent = drive_query_literal(parent_id)
    escaped_name = drive_query_literal(folder_name)
    query = f"name = {escaped_name} and mimeType = 'application/vnd.google-apps.folder' and {escaped_parent} in parents and trashed = false"
    response = service.files().list(q=query, spaces="drive", fields="files(id,name)").execute()
    folders = response.get("files", [])
    if folders:
        return folders[0]
    metadata = {"name": folder_name, "mimeType": "application/vnd.google-apps.folder", "parents": [parent_id]}
    return service.files().create(body=metadata, fields="id,name").execute()


def upload_local_file_to_drive(service, local_path, folder_id, uploaded_name):
    from googleapiclient.http import MediaFileUpload

    media = MediaFileUpload(str(local_path), resumable=True)
    metadata = {"name": uploaded_name, "parents": [folder_id]}
    return service.files().create(body=metadata, media_body=media, fields="id,name").execute()


def backup_notebook05_outputs(reason, include_predictions=False):
    if not BACKUP_NOTEBOOK05_TO_GOOGLE_DRIVE:
        print("Notebook 05 Drive backup skipped; BACKUP_NOTEBOOK05_TO_GOOGLE_DRIVE is False.")
        return None
    service = build_drive_service()
    backup_folder = find_or_create_drive_folder(service, NOTEBOOK05_DRIVE_BACKUP_FOLDER_NAME, DRIVE_PROJECT_FOLDER_ID)
    NOTEBOOK05_STATE["backup_folder_id"] = backup_folder["id"]
    timestamp = pd.Timestamp.utcnow().strftime("%Y%m%dT%H%M%SZ")
    uploaded = []
    for path in OUTPUT_FILES.values():
        if not Path(path).exists() or Path(path).name == OUTPUT_FILES["backup_manifest"].name:
            continue
        uploaded_name = f"{timestamp}__{reason}__{Path(path).name}"
        drive_file = upload_local_file_to_drive(service, path, backup_folder["id"], uploaded_name)
        uploaded.append({"local_path": str(path), "drive_id": drive_file["id"], "drive_name": drive_file["name"]})
    if include_predictions:
        for path in sorted(PREDICTION_DIR.glob("*.npz")):
            uploaded_name = f"{timestamp}__{reason}__predictions__{path.name}"
            drive_file = upload_local_file_to_drive(service, path, backup_folder["id"], uploaded_name)
            uploaded.append({"local_path": str(path), "drive_id": drive_file["id"], "drive_name": drive_file["name"]})
    manifest = {
        "scope": RESULT_SCOPE,
        "created_utc": pd.Timestamp.utcnow().isoformat(),
        "reason": reason,
        "local_output_dir": str(OUTPUT_DIR),
        "backup_folder": backup_folder,
        "uploaded_files": uploaded,
        "run_switches": RUN_SWITCHES,
        "holdout_test_authorized": False,
    }
    with OUTPUT_FILES["backup_manifest"].open("w", encoding="utf-8") as handle:
        json.dump(manifest, handle, indent=2)
    drive_file = upload_local_file_to_drive(
        service,
        OUTPUT_FILES["backup_manifest"],
        backup_folder["id"],
        f"{timestamp}__{reason}__{OUTPUT_FILES['backup_manifest'].name}",
    )
    manifest["backup_manifest_drive_file"] = drive_file
    print("Notebook 05 Drive backup complete:", reason)
    return manifest


def maybe_backup_notebook05_checkpoint(reason, completed_units, include_predictions=False, force=False):
    if not BACKUP_NOTEBOOK05_TO_GOOGLE_DRIVE:
        return None
    completed_units = int(completed_units)
    unit_interval = int(NOTEBOOK05_DRIVE_CHECKPOINT_BACKUP_EVERY_COMPLETED_UNITS)
    minute_interval = float(NOTEBOOK05_DRIVE_CHECKPOINT_BACKUP_EVERY_MINUTES)
    now = pd.Timestamp.utcnow()
    last = NOTEBOOK05_STATE.get("last_drive_checkpoint_utc")
    due_to_units = unit_interval > 0 and completed_units > 0 and completed_units % unit_interval == 0
    due_to_time = last is None or (now - last).total_seconds() >= minute_interval * 60.0
    if not (force or due_to_units or due_to_time):
        return None
    NOTEBOOK05_STATE["last_drive_checkpoint_utc"] = now
    return backup_notebook05_outputs(reason, include_predictions=include_predictions)


def run_notebook05_schema_smoke():
    if HPO_BUDGET != 100 or INNER_FOLD_COUNT != 3:
        raise ValueError("Notebook 05 HPO budget or fold count drifted from protocol.")
    rng = np.random.default_rng(HPO_RNG_SEED)
    trial = sample_lgbm_hpo_params(0, rng)
    if trial["trial_id"] != "lgbm_hpo_000":
        raise ValueError("HPO trial id generation failed.")
    if trial["max_depth"] > 0 and trial["num_leaves"] > 2 ** trial["max_depth"]:
        raise ValueError("Invalid LightGBM HPO constraint passed sampling.")
    dummy_hash = sample_id_hash(["sample_a", "sample_b"])
    if len(dummy_hash) != 64:
        raise ValueError("sample_id_hash did not produce a sha256 digest.")
    print("Notebook 05 schema smoke passed. No model fit was performed.")


## 05S - Schema Smoke

05S checks local schema, search-space sampling, sample-id hashing, and fixed
budget constants. It does not fit any model and does not read holdout/test.


In [ ]:
if RUN_05S_SCHEMA_SMOKE:
    run_notebook05_schema_smoke()
else:
    print("RUN_05S_SCHEMA_SMOKE is False; schema smoke not run.")


## 05A - Notebook 04D Entry Gate

05A reads Notebook 04D artifacts and requires an explicit operator Exit A
acceptance. Passing 05A authorizes LightGBM HPO only; it does not authorize
holdout/test.


In [ ]:
if RUN_05A_04D_ENTRY_GATE:
    entry_decision = assert_notebook05_entry_gate(download_if_missing=True)
    display(pd.DataFrame([entry_decision]))
    backup_notebook05_outputs("completed_05A_04D_entry_gate")
else:
    print("RUN_05A_04D_ENTRY_GATE is False; 04D entry gate not run.")


## Data Loading

This cell resolves the five real ticker files and downloads them through the
Google Drive API when needed. It does not mount MyDrive. For raw `.txt` files it
first scans to the validation boundary and then reads only rows before
`VAL_END`, so closed holdout/test rows are not materialized into the notebook
dataframes. Outputs stay local under `/content/stage0_config_screening_results`.


In [ ]:
RAW_DRIVE_FOLDER_ID = "154SlcH3nViUcvPXFBM-E4NPg_ybljBTG"
RAW_DRIVE_FOLDER_NAME = "s&p 100 adjusted 1 min data"
RAW_DATA_DIR = Path("/content/stage0_raw_stock_data")
DOWNLOAD_RAW_DATA_FROM_DRIVE = True

RAW_DRIVE_FILES = {
    "CSCO": {"name": "CSCO.txt", "file_id": "17A49kUiMELuQqdkOhw1KrpudjP5i5xIN"},
    "JPM": {"name": "JPM.txt", "file_id": "11UQUJKVXTrBb8XFWY5Z8JDQ8_4i_DE-q"},
    "KO": {"name": "KO.txt", "file_id": "1XmtwuZ2dTP20NsU27w5dMyRdSvdnNTSn"},
    "MSFT": {"name": "MSFT.txt", "file_id": "1Ud1SQpQbaiRKemFf9dgu1o_raUPnFvGs"},
    "WMT": {"name": "WMT.txt", "file_id": "1NNfsoUJrrsj2ae5EnC-PTPcZs_QGR_7c"},
}


def is_real_raw_file(path: Path) -> bool:
    if not path.is_file():
        return False
    if path.name.lower().endswith(".gshortcut"):
        return False
    if path.suffix.lower() not in DATA_FILE_SUFFIXES:
        return False
    try:
        return path.stat().st_size > 50_000
    except OSError:
        return False


def build_drive_service():
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
    except ImportError as exc:
        raise RuntimeError(
            "Drive API is unavailable. Open this notebook in Google Colab and "
            "authenticate when prompted; do not use Drive mounting for Stage 0 data."
        ) from exc
    auth.authenticate_user()
    return build("drive", "v3")


def download_raw_drive_files():
    if not DOWNLOAD_RAW_DATA_FROM_DRIVE:
        return
    try:
        from googleapiclient.http import MediaIoBaseDownload
    except ImportError as exc:
        raise RuntimeError("googleapiclient is unavailable in this Colab runtime.") from exc

    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    service = build_drive_service()
    for ticker, item in RAW_DRIVE_FILES.items():
        target = RAW_DATA_DIR / item["name"]
        if is_real_raw_file(target):
            print(f"{ticker}: using cached raw file {target}")
            continue
        print(f"{ticker}: downloading raw Drive file {item['name']} -> {target}")
        request = service.files().get_media(fileId=item["file_id"])
        with target.open("wb") as output:
            downloader = MediaIoBaseDownload(output, request)
            done = False
            while not done:
                _, done = downloader.next_chunk()
        if not is_real_raw_file(target):
            raise ValueError(f"Downloaded file is not a real raw ticker file: {target}")


def resolve_data_files():
    if DOWNLOAD_RAW_DATA_FROM_DRIVE:
        download_raw_drive_files()
    files = {}
    missing = []
    for ticker, item in RAW_DRIVE_FILES.items():
        path = RAW_DATA_DIR / item["name"]
        if is_real_raw_file(path):
            files[ticker] = path
        else:
            missing.append(f"{ticker}: {path}")
    if missing:
        raise FileNotFoundError(
            "Missing required raw ticker files after Drive API resolution: "
            + "; ".join(missing)
        )
    print("resolved raw Drive data files:")
    for ticker, path in files.items():
        print(f"  {ticker}: {path}")
    return files


def find_timestamp_column(columns):
    for candidate in ("timestamp", "datetime", "date", "time"):
        for column in columns:
            if str(column).lower() == candidate:
                return column
    raise ValueError(f"No timestamp-like column found in columns: {list(columns)}")


def normalize_ohlcv_columns(frame, source_name):
    lower_map = {str(column).lower(): column for column in frame.columns}
    rename = {}
    for required in EXPECTED_COLUMNS:
        if required == "timestamp":
            continue
        if required not in lower_map:
            raise ValueError(f"{source_name} missing required column: {required}")
        rename[lower_map[required]] = required
    return frame.rename(columns=rename)


def resample_to_five_minutes(frame):
    resampled = (
        frame.set_index("timestamp")
        .resample("5min")
        .agg({"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"})
        .dropna(subset=["open", "high", "low", "close", "volume"])
        .reset_index()
    )
    times = resampled["timestamp"].dt.time
    return resampled.loc[
        (times >= MARKET_OPEN) & (times <= MARKET_CLOSE),
        list(EXPECTED_COLUMNS),
    ].reset_index(drop=True)


def txt_date_key(date_text):
    parts = str(date_text).strip().split("/")
    if len(parts) != 3:
        raise ValueError(f"Unexpected Date field in raw txt file: {date_text!r}")
    month, day, year = parts
    return int(year), int(month), int(day)


def count_txt_rows_before_validation_end(path):
    validation_end_key = txt_date_key(pd.Timestamp(VAL_END).strftime("%m/%d/%Y"))
    safe_rows = 0
    has_header = False
    reached_boundary = False
    with Path(path).open("rt", encoding="utf-8", errors="replace", newline="") as handle:
        for line in handle:
            stripped = line.strip()
            if not stripped:
                continue
            first_field = stripped.split(",", 1)[0].strip()
            if first_field.lower() == "date":
                has_header = True
                continue
            if txt_date_key(first_field) >= validation_end_key:
                reached_boundary = True
                break
            safe_rows += 1
    if safe_rows == 0:
        raise ValueError(f"No train/validation rows found before {VAL_END} in: {path}")
    if not reached_boundary:
        print(f"{Path(path).name}: no row at or after {VAL_END} found; read capped to file end.")
    return safe_rows, has_header


def read_one_minute_txt(path):
    safe_rows, has_header = count_txt_rows_before_validation_end(path)
    print(f"{Path(path).name}: loading {safe_rows:,} raw one-minute rows before {VAL_END}.")
    frame = pd.read_csv(
        path,
        header=None,
        names=RAW_TXT_COLUMNS,
        nrows=safe_rows,
        skiprows=1 if has_header else None,
        low_memory=False,
    )
    frame = frame.loc[frame["Date"].astype(str).str.lower() != "date"].reset_index(drop=True)
    frame["timestamp"] = pd.to_datetime(
        frame["Date"].astype(str) + " " + frame["Time"].astype(str),
        format="%m/%d/%Y %H:%M",
        errors="raise",
    )
    frame = frame.drop(columns=["Date", "Time"]).rename(
        columns={"Open": "open", "High": "high", "Low": "low", "Close": "close", "Volume": "volume"}
    )
    numeric_columns = ["open", "high", "low", "close", "volume"]
    frame[numeric_columns] = frame[numeric_columns].apply(pd.to_numeric, errors="raise")
    validation_end = pd.Timestamp(VAL_END)
    times = frame["timestamp"].dt.time
    frame = frame.loc[
        (frame["timestamp"] < validation_end)
        & (times >= MARKET_OPEN)
        & (times <= MARKET_CLOSE),
        list(EXPECTED_COLUMNS),
    ]
    return resample_to_five_minutes(frame)


def read_five_minute_csv(path):
    validation_end = pd.Timestamp(VAL_END)
    chunks = []
    for chunk in pd.read_csv(path, chunksize=100_000):
        timestamp_column = find_timestamp_column(chunk.columns)
        chunk = chunk.rename(columns={timestamp_column: "timestamp"}).copy()
        chunk = normalize_ohlcv_columns(chunk, path.name)
        chunk["timestamp"] = pd.to_datetime(chunk["timestamp"], errors="raise")
        raw_chunk_max_timestamp = chunk["timestamp"].max()
        chunk = chunk.loc[chunk["timestamp"] < validation_end, list(EXPECTED_COLUMNS)]
        if not chunk.empty:
            chunks.append(chunk)
        if raw_chunk_max_timestamp >= validation_end:
            break
    if not chunks:
        raise ValueError(f"No train/validation rows found in: {path}")
    return pd.concat(chunks, ignore_index=True)


def load_ticker(ticker, path):
    path = Path(path)
    frame = read_one_minute_txt(path) if path.suffix.lower() == ".txt" else read_five_minute_csv(path)
    frame["ticker"] = ticker
    return frame.sort_values("timestamp").reset_index(drop=True)


RUN_ANY_STAGE = bool(RUN_05B_TRAIN_INNER_HPO or RUN_05D_OFFICIAL_VALIDATION_CONFIRMATION)

if RUN_ANY_STAGE:
    DATA_FILES = resolve_data_files()
    raw_data = {ticker: load_ticker(ticker, DATA_FILES[ticker]) for ticker in TICKERS}

    display(pd.DataFrame([
        {
            "ticker": ticker,
            "rows": len(frame),
            "start": frame["timestamp"].min(),
            "end": frame["timestamp"].max(),
            "source": DATA_FILES[ticker].name,
            "path": str(DATA_FILES[ticker]),
        }
        for ticker, frame in raw_data.items()
    ]))
else:
    DATA_FILES = {}
    raw_data = {}
    print("All Notebook 05 fitting switches are False; data loading skipped.")


## Feature, Label, Split, Scale, Window

These functions implement the active chronology-safe Stage 0 contracts inside a
standalone Colab notebook. They do not import a local helper package or a prior
route. Keep the causal post-bar-close feature rules, cumulative
horizon-return labels, split-boundary invalidation, train-only scaler fitting,
and per-ticker/per-day windows aligned with the active freeze documents before
rerunning Stage 0.

Tabular models use flattened windows: each LogReg/LightGBM sample is
`window_size * n_features`, built from the same per-ticker/per-day rows used by
the sequence models. This makes Stage 0A2 a true window-length sensitivity
check rather than a last-bar sample-count check.

Feature timing boundary: prediction is after the current five-minute bar has
completed, so `close[t]`, `high[t]`, `low[t]`, `volume[t]`, and same-row
timestamp encodings are available. Same-day trailing Bollinger windows may
include `close[t]`; RSI and MACD EWM states are causal but intentionally
continuous across trading days.


In [ ]:
def finite_mask(frame, columns):
    return np.isfinite(frame[list(columns)].to_numpy(dtype=float)).all(axis=1)


def grouped_rolling(series, group_key, window, how):
    grouped = series.groupby(group_key, group_keys=False)
    if how == "mean":
        return grouped.transform(lambda part: part.rolling(window, min_periods=window).mean())
    if how == "std":
        return grouped.transform(lambda part: part.rolling(window, min_periods=window).std())
    raise ValueError(f"Unknown rolling operation: {how}")


def continuous_ewm(series, span):
    return series.ewm(span=span, adjust=False, min_periods=span).mean()


def continuous_wilder_ewm(series, period):
    return series.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean()


def validated_ohlcv(frame):
    out = frame[["open", "high", "low", "close", "volume"]].astype(float)
    bad = (
        (out[["open", "high", "low", "close"]] <= 0).any(axis=1)
        | (out["volume"] < 0)
        | (out["high"] < out["low"])
        | (out["open"] < out["low"])
        | (out["open"] > out["high"])
        | (out["close"] < out["low"])
        | (out["close"] > out["high"])
    )
    if bad.any():
        raise ValueError(f"Invalid OHLCV rows found: {int(bad.sum())}")
    return out


def add_features(frame):
    current = frame.sort_values("timestamp").copy()
    day = current["timestamp"].dt.date
    ohlcv = validated_ohlcv(current)
    close, open_, high, low, volume = (ohlcv[c] for c in ["close", "open", "high", "low", "volume"])

    log_close = np.log(close)
    current["log_return"] = log_close.groupby(day, group_keys=False).diff()
    current["close_to_open_return"] = close / open_ - 1.0
    current["high_low_range"] = np.log(high / low)

    shifted_log_return = current["log_return"].groupby(day, group_keys=False).shift(1)
    current["rolling_volatility_20"] = grouped_rolling(shifted_log_return, day, 20, "std")

    log_volume = np.log1p(volume)
    shifted_log_volume = log_volume.groupby(day, group_keys=False).shift(1)
    current["normalized_volume_20"] = log_volume - grouped_rolling(shifted_log_volume, day, 20, "mean")

    close_delta = close.groupby(day, group_keys=False).diff()
    gain = close_delta.clip(lower=0.0)
    loss = (-close_delta).clip(lower=0.0)
    avg_gain = continuous_wilder_ewm(gain, 14)
    avg_loss = continuous_wilder_ewm(loss, 14)
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    rsi = 100.0 - (100.0 / (1.0 + rs))
    rsi = rsi.mask(avg_loss.eq(0.0) & avg_gain.gt(0.0), 100.0)
    current["rsi_14"] = rsi.mask(avg_loss.eq(0.0) & avg_gain.eq(0.0), 50.0)

    rolling_mean_20 = grouped_rolling(close, day, 20, "mean")
    rolling_std_20 = grouped_rolling(close, day, 20, "std")
    lower_band = rolling_mean_20 - 2.0 * rolling_std_20
    upper_band = rolling_mean_20 + 2.0 * rolling_std_20
    current["bollinger_pctb"] = (close - lower_band) / (upper_band - lower_band).replace(0.0, np.nan)

    ema_12 = continuous_ewm(close, 12)
    ema_26 = continuous_ewm(close, 26)
    macd = ema_12 - ema_26
    signal = continuous_ewm(macd, 9)
    current["normalized_macd_hist"] = (macd - signal) / ema_26.replace(0.0, np.nan)

    minute_of_day = current["timestamp"].dt.hour * 60 + current["timestamp"].dt.minute
    minutes_since_open = minute_of_day - MARKET_OPEN_MINUTE
    current["time_of_day_sin"] = np.sin(2.0 * np.pi * minutes_since_open / TIME_OF_DAY_ENCODING_PERIOD_MINUTES)
    current["time_of_day_cos"] = np.cos(2.0 * np.pi * minutes_since_open / TIME_OF_DAY_ENCODING_PERIOD_MINUTES)
    return current.replace([np.inf, -np.inf], np.nan)


def add_labels(frame, horizon_k, threshold_bps):
    current = frame.sort_values("timestamp").copy()
    threshold = threshold_bps / BPS_TO_DECIMAL
    close = current["close"].astype(float)
    future_timestamp = current["timestamp"].shift(-horizon_k)
    current["future_cumulative_return"] = close.shift(-horizon_k) / close - 1.0

    same_day = pd.Series(True, index=current.index)
    current_day = current["timestamp"].dt.date
    for offset in range(1, horizon_k + 1):
        same_day &= current_day.shift(-offset).eq(current_day)

    actual_horizon = future_timestamp - current["timestamp"]
    expected_horizon = pd.Timedelta(minutes=BAR_INTERVAL_MINUTES * horizon_k)
    current["diagnostic_irregular_horizon"] = (
        future_timestamp.notna() & same_day & actual_horizon.ne(expected_horizon)
    )
    current["invalid_irregular_horizon"] = current["diagnostic_irregular_horizon"]
    current["invalid_missing_future"] = current["future_cumulative_return"].isna()
    current["invalid_cross_day"] = ~same_day
    invalid = (
        current["invalid_missing_future"]
        | current["invalid_cross_day"]
        | current["invalid_irregular_horizon"]
    )
    current["label"] = np.nan
    valid = ~invalid
    current.loc[invalid, "future_cumulative_return"] = np.nan
    current.loc[valid & (current["future_cumulative_return"] > threshold), "label"] = 1
    current.loc[valid & (current["future_cumulative_return"] < -threshold), "label"] = 0
    return current


def assign_split(timestamp):
    ts = pd.Timestamp(timestamp)
    for split_name, (start, end) in CALENDAR_SPLITS.items():
        if pd.Timestamp(start) <= ts < pd.Timestamp(end):
            return split_name
    return "outside_train_validation"


def add_split_and_invalidate_boundaries(frame, horizon_k):
    current = frame.sort_values("timestamp").copy()
    current["split"] = current["timestamp"].map(assign_split)
    horizon_split = current["split"].shift(-horizon_k)
    cross_split = current["future_cumulative_return"].notna() & (current["split"] != horizon_split)
    current["invalid_cross_split"] = cross_split
    current.loc[cross_split, "label"] = np.nan
    current.loc[cross_split, "future_cumulative_return"] = np.nan
    return current


def prepare_split_frames(raw_frames, horizon_k, threshold_bps):
    split_frames = {}
    for ticker, frame in raw_frames.items():
        featured = add_features(frame)
        labeled = add_labels(featured, horizon_k=horizon_k, threshold_bps=threshold_bps)
        split_frames[ticker] = add_split_and_invalidate_boundaries(labeled, horizon_k=horizon_k)
    return split_frames


def fit_transform_train_validation(split_frames, feature_columns):
    train_parts = []
    for frame in split_frames.values():
        train = frame.loc[frame["split"] == "train", list(feature_columns)]
        train_parts.append(train.loc[finite_mask(train, feature_columns)])
    train_matrix = pd.concat(train_parts, axis=0)
    if train_matrix.empty:
        raise ValueError("No train rows available for scaler fit.")

    scaler = StandardScaler().fit(train_matrix)
    scaled = {}
    scaled_columns = [f"{name}_scaled" for name in feature_columns]
    for ticker, frame in split_frames.items():
        current = frame.copy()
        for col in scaled_columns:
            current[col] = np.nan
        eligible = current["split"].isin(["train", "validation"])
        complete = finite_mask(current, feature_columns)
        rows = eligible & complete
        if rows.any():
            current.loc[rows, scaled_columns] = scaler.transform(current.loc[rows, feature_columns])
        scaled[ticker] = current
    return scaled


def build_last_step_windows(frames_by_ticker, feature_columns, split_name, window_size):
    # Flatten the past `window_size` bars into a (window_size * n_features,) vector per sample.
    # Tabular models (LogReg/LightGBM) thus consume lagged history instead of only the last bar,
    # which makes window_size a meaningful search dimension for them too.
    scaled_columns = [f"{name}_scaled" for name in feature_columns]
    n_features = len(feature_columns)
    flat_dim = window_size * n_features
    x_parts, y_parts, owner_parts, timestamp_parts = [], [], [], []
    for ticker, frame in frames_by_ticker.items():
        segment = frame.loc[frame["split"] == split_name].sort_values("timestamp")
        for _, day_frame in segment.groupby(segment["timestamp"].dt.date, sort=True):
            day_frame = day_frame.sort_values("timestamp")
            values = day_frame[scaled_columns].to_numpy(dtype=float)
            labels = day_frame["label"].to_numpy()
            timestamps = day_frame["timestamp"].to_numpy()
            complete_rows = np.isfinite(values).all(axis=1)
            for end_idx in range(window_size - 1, len(day_frame)):
                start_idx = end_idx - window_size + 1
                if not complete_rows[start_idx : end_idx + 1].all():
                    continue
                if pd.isna(labels[end_idx]):
                    continue
                x_parts.append(values[start_idx : end_idx + 1].reshape(-1))
                y_parts.append(int(labels[end_idx]))
                owner_parts.append(ticker)
                timestamp_parts.append(timestamps[end_idx])

    if not x_parts:
        return (
            np.empty((0, flat_dim), dtype=float),
            np.asarray([], dtype=int),
            np.asarray([], dtype=object),
            np.asarray([], dtype="datetime64[ns]"),
        )
    return (
        np.stack(x_parts),
        np.asarray(y_parts, dtype=int),
        np.asarray(owner_parts, dtype=object),
        np.asarray(timestamp_parts, dtype="datetime64[ns]"),
    )


## Notebook 05 Base Helpers

This section copies active Stage 0 metric, dataset, and LightGBM helper definitions. Notebook 05 uses only the LightGBM and dummy paths below.

In [ ]:
DATASET_CACHE = {}


def label_spec(label_config):
    spec = LABEL_CONFIGS[label_config]
    return int(spec["horizon_k"]), float(spec["threshold_bps"])


def subsample_rows_uniformly(x_values, y_values, max_rows, seed=RANDOM_SUBSAMPLE_SEED):
    if max_rows is None or len(y_values) <= max_rows:
        return x_values, y_values
    rng = np.random.default_rng(seed)
    selected = np.sort(rng.choice(len(y_values), size=int(max_rows), replace=False))
    return x_values[selected], y_values[selected]


def subsample_rows_with_owner(x_values, y_values, owner_values, max_rows, seed=RANDOM_SUBSAMPLE_SEED):
    if max_rows is None or len(y_values) <= max_rows:
        return x_values, y_values, owner_values
    rng = np.random.default_rng(seed)
    selected = np.sort(rng.choice(len(y_values), size=int(max_rows), replace=False))
    return x_values[selected], y_values[selected], owner_values[selected]


def evaluate_predictions(y_true, predictions):
    return {
        "macro_f1": float(f1_score(y_true, predictions, labels=[0, 1], average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, predictions)),
        "accuracy": float(accuracy_score(y_true, predictions)),
    }


def dummy_metrics(y_train, y_validation, seed):
    if len(y_train) == 0 or len(y_validation) == 0:
        return {"dummy_macro_f1": np.nan, "dummy_balanced_accuracy": np.nan}
    x_train = np.zeros((len(y_train), 1))
    x_validation = np.zeros((len(y_validation), 1))
    dummy = DummyClassifier(strategy="stratified", random_state=seed).fit(x_train, y_train)
    pred = dummy.predict(x_validation)
    return {
        "dummy_macro_f1": float(f1_score(y_validation, pred, labels=[0, 1], average="macro", zero_division=0)),
        "dummy_balanced_accuracy": float(balanced_accuracy_score(y_validation, pred)),
    }


def sample_std(values):
    values = pd.Series(values).dropna()
    return float(values.std(ddof=1)) if len(values) > 1 else 0.0


T_CRITICAL_ONE_SIDED_95 = {
    2: 6.314,
    3: 2.920,
    4: 2.353,
    5: 2.132,
    6: 2.015,
    7: 1.943,
    8: 1.895,
    9: 1.860,
    10: 1.833,
    11: 1.812,
    12: 1.796,
}


def t_critical_one_sided_95(seed_count):
    if seed_count <= 1:
        return 0.0
    return T_CRITICAL_ONE_SIDED_95.get(int(seed_count), 1.645)


def build_sequence_windows(frames_by_ticker, feature_columns, split_name, window_size):
    scaled_columns = [f"{name}_scaled" for name in feature_columns]
    x_parts, y_parts, owner_parts, timestamp_parts = [], [], [], []
    for ticker, frame in frames_by_ticker.items():
        segment = frame.loc[frame["split"] == split_name].sort_values("timestamp")
        for _, day_frame in segment.groupby(segment["timestamp"].dt.date, sort=True):
            day_frame = day_frame.sort_values("timestamp")
            values = day_frame[scaled_columns].to_numpy(dtype=float)
            labels = day_frame["label"].to_numpy()
            timestamps = day_frame["timestamp"].to_numpy()
            complete_rows = np.isfinite(values).all(axis=1)
            for end_idx in range(window_size - 1, len(day_frame)):
                start_idx = end_idx - window_size + 1
                if not complete_rows[start_idx : end_idx + 1].all():
                    continue
                if pd.isna(labels[end_idx]):
                    continue
                x_parts.append(values[start_idx : end_idx + 1])
                y_parts.append(int(labels[end_idx]))
                owner_parts.append(ticker)
                timestamp_parts.append(timestamps[end_idx])
    if not x_parts:
        return (
            np.empty((0, window_size, len(feature_columns)), dtype=float),
            np.asarray([], dtype=int),
            np.asarray([], dtype=object),
            np.asarray([], dtype="datetime64[ns]"),
        )
    return (
        np.stack(x_parts).astype(np.float32),
        np.asarray(y_parts, dtype=int),
        np.asarray(owner_parts, dtype=object),
        np.asarray(timestamp_parts, dtype="datetime64[ns]"),
    )


def get_dataset(label_config, feature_set, window_size):
    key = (label_config, feature_set, int(window_size))
    if key in DATASET_CACHE:
        dataset = DATASET_CACHE[key].copy()
        dataset["prep_seconds"] = 0.0
        return dataset
    if not raw_data:
        raise RuntimeError("raw_data is empty. Enable RUN_05B_TRAIN_INNER_HPO or RUN_05D_OFFICIAL_VALIDATION_CONFIRMATION and rerun data loading first.")
    horizon_k, threshold_bps = label_spec(label_config)
    feature_columns = FEATURE_SETS[feature_set]
    start = time.perf_counter()
    split_frames = prepare_split_frames(raw_data, horizon_k=horizon_k, threshold_bps=threshold_bps)
    scaled_frames = fit_transform_train_validation(split_frames, feature_columns)
    x_train, y_train, train_owner, train_timestamp = build_last_step_windows(
        scaled_frames, feature_columns, "train", window_size
    )
    x_validation, y_validation, validation_owner, validation_timestamp = build_last_step_windows(
        scaled_frames, feature_columns, "validation", window_size
    )
    x_train_seq, y_train_seq, train_owner_seq, train_timestamp_seq = build_sequence_windows(
        scaled_frames, feature_columns, "train", window_size
    )
    x_validation_seq, y_validation_seq, validation_owner_seq, validation_timestamp_seq = build_sequence_windows(
        scaled_frames, feature_columns, "validation", window_size
    )
    if len(y_train) == 0 or len(y_validation) == 0:
        raise ValueError(f"No tabular windows available for {label_config} / {feature_set} / window={window_size}")
    if len(y_train_seq) != len(y_train) or len(y_validation_seq) != len(y_validation):
        raise ValueError("Tabular and sequence window counts disagree; inspect window construction.")
    dataset = {
        "label_config": label_config,
        "horizon_k": horizon_k,
        "threshold_bps": threshold_bps,
        "feature_set": feature_set,
        "feature_columns": feature_columns,
        "window_size": int(window_size),
        "x_train": x_train,
        "y_train": y_train,
        "train_owner": train_owner,
        "train_timestamp": train_timestamp,
        "train_sample_id": make_notebook05_sample_ids(train_owner, train_timestamp, y_train, "train"),
        "x_validation": x_validation,
        "y_validation": y_validation,
        "validation_owner": validation_owner,
        "validation_timestamp": validation_timestamp,
        "validation_sample_id": make_notebook05_sample_ids(validation_owner, validation_timestamp, y_validation, "validation"),
        "x_train_seq": x_train_seq,
        "y_train_seq": y_train_seq,
        "train_owner_seq": train_owner_seq,
        "train_timestamp_seq": train_timestamp_seq,
        "train_sample_id_seq": make_notebook05_sample_ids(train_owner_seq, train_timestamp_seq, y_train_seq, "train_seq"),
        "x_validation_seq": x_validation_seq,
        "y_validation_seq": y_validation_seq,
        "validation_owner_seq": validation_owner_seq,
        "validation_timestamp_seq": validation_timestamp_seq,
        "validation_sample_id_seq": make_notebook05_sample_ids(validation_owner_seq, validation_timestamp_seq, y_validation_seq, "validation_seq"),
        "prep_seconds": time.perf_counter() - start,
    }
    DATASET_CACHE[key] = dataset.copy()
    return dataset


def fit_predict_logreg(dataset, seed):
    x_train, y_train = subsample_rows_uniformly(dataset["x_train"], dataset["y_train"], MAX_TRAIN_ROWS, seed=seed)
    # Tabular features are flattened windows, so allow more iterations for the
    # higher-dimensional design matrix.
    max_iter = 2000
    model = LogisticRegression(
        solver="liblinear",
        max_iter=max_iter,
        class_weight="balanced",
        random_state=seed,
    )
    start_fit = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(x_train, y_train)
    fit_seconds = time.perf_counter() - start_fit
    start_predict = time.perf_counter()
    pred = model.predict(dataset["x_validation"])
    predict_seconds = time.perf_counter() - start_predict
    convergence_warnings = [w for w in caught if issubclass(w.category, ConvergenceWarning)]
    max_iter_reached = bool((model.n_iter_ >= max_iter).any())
    fit_status = "converged" if not convergence_warnings and not max_iter_reached else "convergence_warning"
    return pred, fit_seconds, predict_seconds, int(len(y_train)), fit_status


def fit_predict_lightgbm(dataset, seed):
    lgb = ensure_lightgbm()
    x_train, y_train = subsample_rows_uniformly(dataset["x_train"], dataset["y_train"], MAX_TRAIN_ROWS, seed=seed)
    model = lgb.LGBMClassifier(
        **LGBM_PARAMS,
        class_weight="balanced",
        random_state=seed,
        verbosity=-1,
    )
    start_fit = time.perf_counter()
    model.fit(x_train, y_train)
    fit_seconds = time.perf_counter() - start_fit
    start_predict = time.perf_counter()
    pred = model.predict(dataset["x_validation"])
    predict_seconds = time.perf_counter() - start_predict
    return pred, fit_seconds, predict_seconds, int(len(y_train)), "not_applicable"


def set_global_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch = ensure_torch()
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    return torch


def make_simple_gru(input_dim, seed):
    torch = set_global_seed(seed)
    nn = torch.nn

    class SimpleGRUClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.gru = nn.GRU(input_dim, 32, num_layers=1, batch_first=True)
            self.dropout = nn.Dropout(TORCH_DROPOUT)
            self.head = nn.Linear(32, 2)

        def forward(self, x):
            output, _ = self.gru(x)
            return self.head(self.dropout(output[:, -1, :]))

    return SimpleGRUClassifier()


def make_ms_dlinear_tcn(input_dim, window_size, seed):
    torch = set_global_seed(seed)
    nn = torch.nn
    functional = torch.nn.functional

    class CausalConvBlock(nn.Module):
        def __init__(self, in_channels, out_channels, kernel_size, dropout):
            super().__init__()
            self.pad = kernel_size - 1
            self.conv = nn.Conv1d(in_channels, out_channels, kernel_size)
            self.norm = nn.BatchNorm1d(out_channels)
            self.dropout = nn.Dropout(dropout)
            self.proj = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

        def forward(self, x):
            residual = self.proj(x)
            padded = functional.pad(x, (self.pad, 0))
            out = self.conv(padded)
            out = self.dropout(torch.relu(self.norm(out)))
            return out + residual

    class MultiScaleDLinearTCNClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.tcn = nn.Sequential(
                CausalConvBlock(input_dim, TORCH_TCN_CHANNELS[0], TORCH_TCN_KERNEL_SIZE, TORCH_DROPOUT),
                CausalConvBlock(TORCH_TCN_CHANNELS[0], TORCH_TCN_CHANNELS[1], TORCH_TCN_KERNEL_SIZE, TORCH_DROPOUT),
            )
            self.scale_head = nn.Linear(input_dim * len(TORCH_MOVING_AVG_KERNELS), 16)
            self.head = nn.Linear(TORCH_TCN_CHANNELS[-1] + 16, 2)

        def moving_average_last(self, x, kernel):
            pad = kernel - 1
            padded = functional.pad(x.transpose(1, 2), (pad, 0), mode="replicate")
            avg = functional.avg_pool1d(padded, kernel_size=kernel, stride=1)
            return avg[:, :, -1]

        def forward(self, x):
            tcn_last = self.tcn(x.transpose(1, 2))[:, :, -1]
            scale_parts = [self.moving_average_last(x, kernel) for kernel in TORCH_MOVING_AVG_KERNELS]
            scale = torch.relu(self.scale_head(torch.cat(scale_parts, dim=1)))
            return self.head(torch.cat([tcn_last, scale], dim=1))

    return MultiScaleDLinearTCNClassifier()


def run_torch_shape_smoke(input_dim, window_size):
    torch = ensure_torch()
    for model_name, factory in (
        ("simple_gru", lambda: make_simple_gru(input_dim, 41)),
        ("ms_dlinear_tcn", lambda: make_ms_dlinear_tcn(input_dim, window_size, 41)),
    ):
        model = factory()
        x = torch.zeros((2, window_size, input_dim), dtype=torch.float32)
        y = torch.tensor([0, 1], dtype=torch.long)
        logits = model(x)
        if tuple(logits.shape) != (2, 2):
            raise ValueError(f"{model_name} shape smoke failed: logits shape {tuple(logits.shape)}")
        loss = torch.nn.CrossEntropyLoss()(logits, y)
        loss.backward()
    print("Deep adapter shape smoke passed.")


def fit_predict_torch_sequence(dataset, seed, model_name):
    torch = set_global_seed(seed)
    x_train, y_train, train_owner = subsample_rows_with_owner(
        dataset["x_train_seq"],
        dataset["y_train_seq"],
        dataset["train_owner_seq"],
        MAX_TRAIN_ROWS,
        seed=seed,
    )
    x_validation = dataset["x_validation_seq"]
    input_dim = x_train.shape[-1]
    window_size = x_train.shape[1]
    if model_name == "simple_gru":
        model = make_simple_gru(input_dim, seed)
    elif model_name == "ms_dlinear_tcn":
        model = make_ms_dlinear_tcn(input_dim, window_size, seed)
    else:
        raise ValueError(f"Unknown torch model: {model_name}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    train_x_tensor = torch.tensor(x_train, dtype=torch.float32)
    train_y_tensor = torch.tensor(y_train, dtype=torch.long)
    counts = np.bincount(y_train, minlength=2).astype(float)
    class_weights = counts.sum() / np.maximum(counts, 1.0)
    class_weights = class_weights / class_weights.mean()
    criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=TORCH_LEARNING_RATE, weight_decay=TORCH_WEIGHT_DECAY)
    generator = torch.Generator().manual_seed(seed)
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(train_x_tensor, train_y_tensor),
        batch_size=TORCH_BATCH_SIZE,
        shuffle=True,
        generator=generator,
    )

    start_fit = time.perf_counter()
    model.train()
    for _ in range(TORCH_EPOCHS):
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
    fit_seconds = time.perf_counter() - start_fit

    start_predict = time.perf_counter()
    model.eval()
    preds = []
    with torch.no_grad():
        for start in range(0, len(x_validation), TORCH_BATCH_SIZE):
            batch = torch.tensor(x_validation[start : start + TORCH_BATCH_SIZE], dtype=torch.float32, device=device)
            preds.append(model(batch).argmax(dim=1).cpu().numpy())
    predict_seconds = time.perf_counter() - start_predict
    return np.concatenate(preds), fit_seconds, predict_seconds, int(len(y_train)), "fixed_epochs_no_early_stopping"


def fit_predict_model(dataset, model_name, seed):
    if model_name == "logreg":
        return fit_predict_logreg(dataset, seed)
    if model_name == "lightgbm":
        return fit_predict_lightgbm(dataset, seed)
    if model_name in {"simple_gru", "ms_dlinear_tcn"}:
        return fit_predict_torch_sequence(dataset, seed, model_name)
    raise ValueError(f"Unknown model: {model_name}")


def concentration_from_per_ticker(per_ticker_rows):
    deltas = [row["per_ticker_delta_macro_f1_vs_dummy"] for row in per_ticker_rows]
    positive = [float(delta) for delta in deltas if pd.notna(delta) and delta > 0]
    positive_ticker_count = int(len(positive))
    top_ticker_gain_share = float(max(positive) / sum(positive)) if positive else 0.0
    return positive_ticker_count, top_ticker_gain_share


def run_one_model_seed(stage, model_name, label_config, feature_set, window_size, seed):
    dataset = get_dataset(label_config, feature_set, window_size)
    prep_seconds = float(dataset["prep_seconds"])
    pred, fit_seconds, predict_seconds, train_n, fit_status = fit_predict_model(dataset, model_name, seed)
    pooled_metrics = evaluate_predictions(dataset["y_validation"], pred)
    pooled_dummy = dummy_metrics(dataset["y_train"], dataset["y_validation"], seed)
    per_ticker_rows = []
    for ticker in TICKERS:
        val_mask = dataset["validation_owner"] == ticker
        train_mask = dataset["train_owner"] == ticker
        if not val_mask.any():
            continue
        ticker_metrics = evaluate_predictions(dataset["y_validation"][val_mask], pred[val_mask])
        ticker_dummy = dummy_metrics(dataset["y_train"][train_mask], dataset["y_validation"][val_mask], seed)
        per_ticker_rows.append({
            "stage": stage,
            "model": model_name,
            "label_config": label_config,
            "horizon_k": dataset["horizon_k"],
            "threshold_bps": dataset["threshold_bps"],
            "feature_set": feature_set,
            "window_size": int(window_size),
            "seed": int(seed),
            "scope": RESULT_SCOPE,
            "macro_f1": ticker_metrics["macro_f1"],
            "balanced_accuracy": ticker_metrics["balanced_accuracy"],
            "accuracy": ticker_metrics["accuracy"],
            "dummy_macro_f1": ticker_dummy["dummy_macro_f1"],
            "dummy_balanced_accuracy": ticker_dummy["dummy_balanced_accuracy"],
            "delta_macro_f1_vs_dummy": ticker_metrics["macro_f1"] - ticker_dummy["dummy_macro_f1"],
            "delta_balanced_accuracy_vs_dummy": (
                ticker_metrics["balanced_accuracy"] - ticker_dummy["dummy_balanced_accuracy"]
            ),
            "n": int(val_mask.sum()),
            "ticker_or_pooled": ticker,
            "prep_seconds": prep_seconds,
            "fit_seconds": float(fit_seconds),
            "predict_seconds": float(predict_seconds),
            "total_seconds": prep_seconds + float(fit_seconds) + float(predict_seconds),
            "per_ticker_delta_macro_f1_vs_dummy": ticker_metrics["macro_f1"] - ticker_dummy["dummy_macro_f1"],
            "positive_ticker_count": np.nan,
            "top_ticker_gain_share": np.nan,
            "train_n": int(train_mask.sum()),
            "fit_status": fit_status,
        })
    positive_ticker_count, top_ticker_gain_share = concentration_from_per_ticker(per_ticker_rows)
    for row in per_ticker_rows:
        row["positive_ticker_count"] = positive_ticker_count
        row["top_ticker_gain_share"] = top_ticker_gain_share
    pooled_row = {
        "stage": stage,
        "model": model_name,
        "label_config": label_config,
        "horizon_k": dataset["horizon_k"],
        "threshold_bps": dataset["threshold_bps"],
        "feature_set": feature_set,
        "window_size": int(window_size),
        "seed": int(seed),
        "scope": RESULT_SCOPE,
        "macro_f1": pooled_metrics["macro_f1"],
        "balanced_accuracy": pooled_metrics["balanced_accuracy"],
        "accuracy": pooled_metrics["accuracy"],
        "dummy_macro_f1": pooled_dummy["dummy_macro_f1"],
        "dummy_balanced_accuracy": pooled_dummy["dummy_balanced_accuracy"],
        "delta_macro_f1_vs_dummy": pooled_metrics["macro_f1"] - pooled_dummy["dummy_macro_f1"],
        "delta_balanced_accuracy_vs_dummy": pooled_metrics["balanced_accuracy"] - pooled_dummy["dummy_balanced_accuracy"],
        "n": int(len(dataset["y_validation"])),
        "ticker_or_pooled": "pooled",
        "prep_seconds": prep_seconds,
        "fit_seconds": float(fit_seconds),
        "predict_seconds": float(predict_seconds),
        "total_seconds": prep_seconds + float(fit_seconds) + float(predict_seconds),
        "per_ticker_delta_macro_f1_vs_dummy": np.nan,
        "positive_ticker_count": positive_ticker_count,
        "top_ticker_gain_share": top_ticker_gain_share,
        "train_n": int(train_n),
        "fit_status": fit_status,
    }
    return pooled_row, per_ticker_rows


def run_stage_grid(stage, specs):
    pooled_rows = []
    per_ticker_rows = []
    for spec in specs:
        print(
            stage,
            spec["model"],
            spec["label_config"],
            spec["feature_set"],
            "window",
            spec["window_size"],
            "seed",
            spec["seed"],
        )
        pooled, per_ticker = run_one_model_seed(
            stage=stage,
            model_name=spec["model"],
            label_config=spec["label_config"],
            feature_set=spec["feature_set"],
            window_size=spec["window_size"],
            seed=spec["seed"],
        )
        pooled_rows.append(pooled)
        per_ticker_rows.extend(per_ticker)
    return pd.DataFrame(pooled_rows), pd.DataFrame(per_ticker_rows)


def summarize_pooled(pooled):
    if pooled.empty:
        return pd.DataFrame()
    rows = []
    keys = ["stage", "model", "label_config", "horizon_k", "threshold_bps", "feature_set", "window_size", "scope"]
    for key_values, group in pooled.groupby(keys, sort=False):
        record = dict(zip(keys, key_values))
        seed_count = int(group["seed"].nunique())
        macro_std = sample_std(group["macro_f1"])
        bal_std = sample_std(group["balanced_accuracy"])
        record.update({
            "seed_count": seed_count,
            "macro_f1_mean": float(group["macro_f1"].mean()),
            "macro_f1_std": macro_std,
            "macro_f1_lcb_95": float(
                group["macro_f1"].mean()
                - t_critical_one_sided_95(seed_count) * macro_std / math.sqrt(max(seed_count, 1))
            ),
            "balanced_accuracy_mean": float(group["balanced_accuracy"].mean()),
            "balanced_accuracy_std": bal_std,
            "dummy_macro_f1_mean": float(group["dummy_macro_f1"].mean()),
            "dummy_balanced_accuracy_mean": float(group["dummy_balanced_accuracy"].mean()),
            "delta_macro_f1_vs_dummy_mean": float(group["delta_macro_f1_vs_dummy"].mean()),
            "delta_balanced_accuracy_vs_dummy_mean": float(group["delta_balanced_accuracy_vs_dummy"].mean()),
            "n_mean": float(group["n"].mean()),
            "positive_ticker_count": int(round(group["positive_ticker_count"].mean())),
            "top_ticker_gain_share": float(group["top_ticker_gain_share"].mean()),
            "prep_seconds_mean": float(group["prep_seconds"].mean()),
            "fit_seconds_mean": float(group["fit_seconds"].mean()),
            "predict_seconds_mean": float(group["predict_seconds"].mean()),
            "total_seconds_mean": float(group["total_seconds"].mean()),
        })
        record["basic_gate"] = bool(
            record["delta_macro_f1_vs_dummy_mean"] > 0
            and record["macro_f1_lcb_95"] > record["dummy_macro_f1_mean"]
        )
        record["lcb_eligible"] = bool(
            record["basic_gate"]
            and record["delta_balanced_accuracy_vs_dummy_mean"] > 0
            and record["top_ticker_gain_share"] < 0.50
            and record["positive_ticker_count"] >= 3
        )
        rows.append(record)
    return pd.DataFrame(rows)


def tuple_from_row(row, include_window):
    if row is None:
        return None
    if include_window:
        return {
            "label_config": row["label_config"],
            "feature_set": row["feature_set"],
            "window_size": int(row["window_size"]),
        }
    return {"label_config": row["label_config"], "feature_set": row["feature_set"]}


def select_candidates(summary, include_window):
    if summary.empty or not summary["basic_gate"].any():
        return {
            "stage0_result": "do_not_decide_config",
            "mean_candidate": None,
            "lcb_candidate": None,
            "candidate_count": 0,
        }
    basic = summary.loc[summary["basic_gate"]].sort_values("macro_f1_mean", ascending=False)
    lcb = summary.loc[summary["lcb_eligible"]].sort_values("macro_f1_lcb_95", ascending=False)
    mean_candidate = tuple_from_row(basic.iloc[0], include_window=include_window)
    lcb_candidate = tuple_from_row(lcb.iloc[0], include_window=include_window) if not lcb.empty else None
    unique_candidates = []
    for candidate in (mean_candidate, lcb_candidate):
        if candidate is not None and candidate not in unique_candidates:
            unique_candidates.append(candidate)
    return {
        "stage0_result": "candidate_config_selected",
        "mean_candidate": mean_candidate,
        "lcb_candidate": lcb_candidate,
        "candidate_count": len(unique_candidates),
        "candidates": unique_candidates,
    }


def write_outputs(pooled, per_ticker, summary, file_keys):
    pooled.to_csv(OUTPUT_FILES[file_keys[0]], index=False)
    per_ticker.to_csv(OUTPUT_FILES[file_keys[1]], index=False)
    if summary is not None:
        summary.to_csv(OUTPUT_FILES[file_keys[2]], index=False)
    print("wrote", [str(OUTPUT_FILES[key]) for key in file_keys])


## 05B - Train-Inner Chronological LightGBM HPO

05B runs exactly 100 random-search LightGBM trials across 3 chronological
train-inner folds. It does not use official validation rows for HPO.


In [ ]:
if RUN_05B_TRAIN_INNER_HPO:
    hpo_search_manifest, inner_fold_manifest, inner_hpo_results, inner_hpo_summary = run_train_inner_hpo_05b()
    display(inner_hpo_summary.sort_values("inner_lcb_macro_f1", ascending=False).head(10))
    backup_notebook05_outputs("completed_05B_train_inner_hpo")
else:
    print("RUN_05B_TRAIN_INNER_HPO is False; train-inner HPO not run.")


## 05C - Finalist Selection

05C selects finalists only from train-inner HPO results. The rank-1 finalist is
the `train_inner_winner`. Official validation cannot replace it.


In [ ]:
if RUN_05C_SELECT_FINALISTS:
    finalists = select_finalists_05c()
    display(finalists)
    backup_notebook05_outputs("completed_05C_select_finalists")
else:
    print("RUN_05C_SELECT_FINALISTS is False; finalists not selected.")


## 05D - Official Validation Confirmation

05D evaluates default LightGBM, the train-inner winner, the remaining finalists
up to `N_FINALISTS`, and same-row dummy baselines on official validation. It
does not use official validation for early stopping or profile selection.


In [ ]:
if RUN_05D_OFFICIAL_VALIDATION_CONFIRMATION:
    official_pooled, official_per_ticker, official_summary = run_official_validation_confirmation_05d()
    display(official_summary)
    backup_notebook05_outputs("completed_05D_official_validation_confirmation", include_predictions=True)
else:
    print("RUN_05D_OFFICIAL_VALIDATION_CONFIRMATION is False; official validation confirmation not run.")


## 05E - Decision Record

05E writes the validation-only decision record and allowed wording. It does not
authorize holdout/test and does not choose any selective confidence threshold.


In [ ]:
if RUN_05E_DECISION_RECORD:
    decision_record = write_decision_record_05e()
    display(pd.DataFrame([decision_record]))
    backup_notebook05_outputs("completed_05E_decision_record")
else:
    print("RUN_05E_DECISION_RECORD is False; decision record not written.")


## Interpretation Boundary

Notebook 05 is `validation_only`.

Allowed wording:

```text
Notebook 05 performs validation-only LightGBM hyperparameter tuning under a
train-inner chronological HPO design. The official validation partition is used
only to confirm the pre-selected train-inner winner and a fixed number of
finalists.
```

```text
The selected tuned configuration is the train-inner HPO winner, not the
official-validation-best finalist.
```

```text
Notebook 05 does not authorize holdout/test evaluation.
```

Forbidden wording:

```text
The tuned model is final.
The tuned model is holdout-ready.
The tuned model significantly beats LogReg.
The tuned model proves LightGBM is superior to deep learning.
The official-validation-best finalist is selected.
Selective coverage is now the final trading threshold.
```

Notebook 06 is reserved for separately pre-registered prediction-time
abstention / high-confidence coverage calibration. Notebook 05 does not select
coverage thresholds.
